# AF / DepMap / HSA / GLM-5 — one comparable frame

Every arm from the HSA strand and the AF + DepMap strand in **one long table**, so any metric
can be plotted against any other and any subset of arms can be selected per figure.

**What is here**

| strand | arms |
|---|---|
| AF alone, AF + DepMap | from the benchmark sweeps under `output/runs/<method>/` |
| HSA, DepMap + HSA (four DepMap weights), AF + HSA, AF + DepMap + HSA | from `output/runs/external/*_hsa*.json` |
| GLM-5 warm start -> AF alone / AF + DepMap / AF + DepMap + screen-kNN | from the `*-hyb-glm5*` sweeps |
| *(later)* GLM-5 warm start -> HSA | **one registry row**, see the last cell |

**How to choose what a figure shows.** Every figure takes `select=`. A term matches, in this
order: exact canonical label, exact **group**, the shorthands `all`/`hsa`/`glm5`/`ours`, then
substring. `DEFAULT_SELECT` in the setup cell changes every figure at once.
`list_arms()` prints every arm as a copy-pasteable Python list — copy the block, delete the
lines you do not want, pass the rest as `select=`.

**Metrics.** `ef` (adjusted EF), `hits`, `hit_rate` (hits / measured picks), `rate_slots`
(hits / slots offered — the gap to `hit_rate` is the LLM shortfall), `recall`
(hits / hits available). Every figure takes `metric=`.

**Read §7 of `AF_DEPMAP_HSA_SUMMARY.md` before quoting anything from here.** In particular EF
and hits rank these arms in *opposite* orders, and the screen filters below change the answer.

In [ ]:
# --- setup -------------------------------------------------------------------
import sys, json, re, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "assayloop").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts" / "diagnostics"))

from assayloop.tasks import load_screens
from run_paths import sweep_root as _sweep_root

RUNS = ROOT / "output" / "runs"
EXT  = RUNS / "external"
REPRO = ROOT / "_reproduction"
BUDGET = 1000

def sweep_base(sid):
    """`output/runs` is sorted into one folder per method, so resolve a sweep id to it."""
    return _sweep_root(RUNS, sid)

# ---------------------------------------------------------------- screen filters
# THESE CHANGE THE ANSWER. The 218-screen validation fold has median n_genes 2,318 and 98
# screens under 1,000 genes, where EF is meaningless because the budget exceeds the
# library. The curated 20-screen sets apply `num_hits_min: 50` and the fold does not --
# which is the whole reason the curated and fold verdicts disagreed (HANDOFF 24.19).
#
#   CURATED_CRITERIA = True reproduces the curated sets' own selection_criteria block
#   and is the FAIR comparison when putting fold numbers beside curated ones.
GENOME_SCALE_ONLY = True    # drop libraries at or below the 1000-pick budget
CURATED_CRITERIA  = False   # 18000 <= n_genes <= 22000, total_hits >= 50, rate <= 0.15
MIN_GENES, MAX_GENES = 0, 0 # 0 = off; overridden by CURATED_CRITERIA
MIN_HITS, MAX_HIT_RATE = 0, 0.0

def _criteria():
    if CURATED_CRITERIA:
        return dict(min_genes=18000, max_genes=22000, min_hits=50, max_rate=0.15)
    return dict(min_genes=MIN_GENES, max_genes=MAX_GENES,
                min_hits=MIN_HITS, max_rate=MAX_HIT_RATE)

# ---------------------------------------------------------------- aggregation unit
# Screens from one publication are near-duplicates, so a screen-level mean and bootstrap
# over-weight whichever paper happens to contribute most rows. paper_validation is 20
# screens over 10 publications with ONE supplying 8 (the Vichas U_179x family, mean EF
# 11.05 against 3.47 elsewhere); the 218-fold is 34 publications, largest cluster 86.
#   CLUSTER_BY = "publication"  each publication contributes the mean of its screens
#   CLUSTER_BY = "screen"       every screen counts once (what the paper reports)
CLUSTER_BY = "publication"
N_BOOT = 20000
RNG_SEED = 0

# ---------------------------------------------------------------- selection
# ONE PLACE TO CHANGE WHAT EVERY FIGURE SHOWS. `list_arms()` prints the full menu.
SELECT = [
    "AF alone",
    "AF + DepMap (dm=0.5)",
    "HSA only [real mean]",
    "DepMap + HSA (dm=0.25) [real mean]",
    "DepMap + HSA (dm=0.5) [real mean]",
]
DEFAULT_SELECT = SELECT      # or "all", or None for each figure's own default

STANDARD = "AF + DepMap (dm=0.5)"   # outlined heavily on bar charts
MAX_ARMS_PER_FIGURE = 8             # the palette holds 8 hues; a 9th repeats and lies

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3",
           "#937860", "#DA8BC3", "#8C8C8C"]
NEUTRAL = "#B0B0B0"
print("setup ok — ROOT:", ROOT)

## 1. The registry

Two kinds of source, normalised into the same rows.

**External JSONs** (`output/runs/external/*_hsa*.json`) already hold one row per
(arm, screen) with `hits`, `ef`, `n_genes`, `total_hits`, `n_measured`, `picks`. Their arm
names are mapped to weight triples by `ARM_WEIGHTS`, and the **canonical label is built from
the weights**, so `DepMap + HSA dm.50` and `DepMap + HSA (no AF term)` — identical settings
under two names — collapse to one arm.

**Sweeps** (`output/runs/<method>/<sweep>-NN-<screen>/result.json`) are read from
`final_metrics`, which carries `n_hits`, `n_hits_vs_random` (the adjusted EF) and
`total_hits` — the same quantities, so no metric is recomputed differently for the two
sources.

`verify_sources()` reports anything missing instead of raising, so a partial checkout still
plots what it has.

In [ ]:
# --- the registry ------------------------------------------------------------
# arm name as RECORDED  ->  (af_weight, depmap_weight, hsa_weight)
ARM_WEIGHTS = {
    "AF alone":                   (1.0, 0.00, 0.0),
    "AF alone (greedy)":          (1.0, 0.00, 0.0),
    "AF + DepMap":                (1.0, 0.50, 0.0),
    "AF + HSA":                   (1.0, 0.00, 0.5),
    "AF + DepMap + HSA":          (1.0, 0.50, 0.5),
    "AF + DepMap + HSA .25":      (1.0, 0.50, 0.25),
    "HSA only (ours)":            (0.0, 0.00, 1.0),
    "DepMap + HSA (no AF term)":  (0.0, 0.50, 1.0),
    "HSA + DepMap eq":            (0.0, 1.00, 1.0),
    "HSA + DepMap dm2":           (0.0, 2.00, 1.0),
    "DepMap + HSA dm.25":         (0.0, 0.25, 1.0),
    "DepMap + HSA dm.50":         (0.0, 0.50, 1.0),
    "DepMap + HSA dm.75":         (0.0, 0.75, 1.0),
    "DepMap + HSA dm1.0":         (0.0, 1.00, 1.0),
}

def canon_label(af, dm, hsa, mean_mode=None, prefix=None, suffix=None):
    """One label per weight triple, so aliases collapse and the weight axis is readable."""
    if prefix:
        base = prefix
    elif hsa == 0 and dm == 0:
        base = "AF alone"
    elif hsa == 0:
        base = f"AF + DepMap (dm={dm:g})"
    elif af == 0 and dm == 0:
        base = "HSA only"
    elif af == 0:
        base = f"DepMap + HSA (dm={dm:g})"
    elif dm == 0:
        base = f"AF + HSA (hsa={hsa:g})"
    else:
        base = f"AF + DepMap + HSA (dm={dm:g}, hsa={hsa:g})"
    if suffix:
        base = f"{base} {suffix}"
    if hsa > 0 and mean_mode in ("zero", "real"):
        base = f"{base} [{'real mean' if mean_mode == 'real' else 'mean=0'}]"
    return base

def arm_group(af, dm, hsa, prefix=None):
    if prefix:
        return "GLM-5"
    if hsa > 0 and af == 0 and dm > 0: return "DepMap+HSA"
    if hsa > 0 and af == 0:            return "HSA"
    if hsa > 0:                        return "AF+HSA"
    if dm > 0:                         return "AF+DepMap"
    return "AF"

# ---- external per-screen JSONs: (screen set, file, mean mode) -------------------
EXTERNAL_SOURCES = [
    ("paper_test",       "paper_test_hsa.json",                 "zero"),
    ("paper_test",       "paper_test_hsa_noaf.json",            "zero"),
    ("paper_test",       "paper_test_hsa_realmean.json",        "real"),
    ("paper_validation", "paper_validation_hsa.json",           "zero"),
    ("paper_validation", "paper_validation_hsa_realmean.json",  "real"),
    ("validation",       "validation_fold_hsa.json",            "zero"),
    ("validation",       "validation_fold_hsa_realmean.json",   "real"),
    ("bundle18",         "bundle18_sixarm_results.json",        "zero"),
    ("bundle18",         "bundle18_results.json",               "zero"),
]

# The 334-screen TEST fold, off by default. Turning it on adds 668 rows to LONG and so a
# `test` panel to every figure that facets by screen set -- which is why it is a switch and
# not just more registry rows. AF alone and AF + DepMap already exist there as sweeps; HSA
# does NOT (see the Figure 8 cell for what producing it costs).
INCLUDE_TEST_FOLD = False

# ---- sweeps: (screen set, sweep id, label prefix or None, af, dm, hsa) ---------
# `prefix=None` means the label is BUILT FROM THE WEIGHTS by canon_label, so a sweep arm
# and an external arm with the same settings land on the same row of every figure.
SWEEP_SOURCES = [
    ("paper_test",       "sweep-dm-pt-w0",            None, 1.0, 0.0, 0.0),
    ("paper_test",       "sweep-dmne-cd-pt",          None, 1.0, 0.5, 0.0),
    ("paper_validation", "sweep-cdv-w0",              None, 1.0, 0.0, 0.0),
    ("paper_validation", "sweep-dmne-cd-pv",          None, 1.0, 0.5, 0.0),
    ("validation",       "sweep-dm-validation-w0",    None, 1.0, 0.0, 0.0),
    ("validation",       "sweep-dmne-cd-validation",  None, 1.0, 0.5, 0.0),
    # the test fold: both baselines exist at 334 screens. `sweep-tf-test-w0` loads the FULL
    # atlas but at weight=0, where the DepMap term never enters the fusion, so it is
    # bit-identical to plain AssayFormer and is NOT a cross-atlas comparison.
    ("test",             "sweep-tf-test-w0",          None, 1.0, 0.0, 0.0),
    ("test",             "sweep-dmne-cd-test",        None, 1.0, 0.5, 0.0),
    # ---- GLM-5 warm start, handed to each receiving method (HANDOFF 4.61) ----
    ("paper_test",       "sweep-glm5tu-pt-af",
     "GLM-5 +top-up -> AF alone",                         1.0, 0.0, 0.0),
    ("paper_test",       "sweep-dmcd-hyb-glm5tu-pt",
     "GLM-5 +top-up -> AF + DepMap",                      1.0, 0.5, 0.0),
    ("paper_test",       "sweep-dmne-hyb-glm5tu-pt",
     "GLM-5 +top-up -> AF + DepMap + screen-kNN",         1.0, 0.5, 0.0),
    ("paper_test",       "sweep-dmne-hyb-glm5-pt",
     "GLM-5 -> AF + DepMap + screen-kNN",                 1.0, 0.5, 0.0),
    ("paper_validation", "sweep-dmcd-hyb-glm5tu-pv",
     "GLM-5 +top-up -> AF + DepMap",                      1.0, 0.5, 0.0),
    ("paper_validation", "sweep-dmne-hyb-glm5tu-pv",
     "GLM-5 +top-up -> AF + DepMap + screen-kNN",         1.0, 0.5, 0.0),
    ("paper_validation", "sweep-dmne-hyb-glm5-pv",
     "GLM-5 -> AF + DepMap + screen-kNN",                 1.0, 0.5, 0.0),
]

def active_sources():
    """The registry minus anything the switches turn off."""
    ext = [r for r in EXTERNAL_SOURCES if INCLUDE_TEST_FOLD or r[0] != "test"]
    swp = [r for r in SWEEP_SOURCES if INCLUDE_TEST_FOLD or r[0] != "test"]
    return ext, swp


def verify_sources():
    rows = []
    ext, swp = active_sources()
    for sset, fn, mm in ext:
        p = EXT / fn
        rows.append(dict(kind="external", sset=sset, name=fn, mean=mm,
                         found=p.exists(),
                         n=len(json.loads(p.read_text())) if p.exists() else 0))
    for sset, sid, pre, af, dm, hsa in swp:
        try:
            base = sweep_base(sid)
            dirs = sorted(base.glob(f"{sid}-*"))
        except Exception:
            dirs = []
        rows.append(dict(kind="sweep", sset=sset, name=sid, mean="n/a",
                         found=bool(dirs), n=len(dirs)))
    V = pd.DataFrame(rows)
    miss = V[~V.found]
    if len(miss):
        print("MISSING — these are skipped, every figure still draws the rest:")
        print(miss[["kind", "sset", "name"]].to_string(index=False))
    else:
        print("all sources present")
    return V

SRC = verify_sources()
SRC

## 2. Loaders and the master table

`LONG` is the only table the figures read. One row per **(arm, screen set, screen)**:

| column | meaning |
|---|---|
| `label`, `group` | canonical arm name and its family |
| `af_w`, `dm_w`, `hsa_w` | the three RRF weights — the axis for the weight-sweep figure |
| `mean_mode` | `real` / `zero` for HSA arms, `n/a` otherwise |
| `sset`, `screen` | screen set and screen id |
| `hits`, `ef` | unique measured hits, adjusted EF |
| `n_measured`, `picks`, `n_genes`, `total_hits` | denominators |
| `pub` | publication id — the clustering unit |
| `hit_rate`, `rate_slots`, `recall` | derived |

Screen metadata (library size, hits available, perturbation type, phenotype class,
publication) comes from `load_screens` and the manifests, **not** from the run files, so it is
identical across arms by construction.

In [ ]:
# --- loaders -----------------------------------------------------------------
_SC = {}
def screens_for(sset):
    """ScreenRecords for a screen set. `bundle18` comes from the reproduction bundle."""
    if sset in _SC:
        return _SC[sset]
    if sset == "bundle18":
        from assaybench.data.screens import ScreenRecord
        out = []
        for p in sorted((REPRO / "bundle" / "screens").glob("*.json")):
            out.append(ScreenRecord(**json.loads(p.read_text())))
        _SC[sset] = out
    else:
        _SC[sset] = load_screens(target_set=sset)
    return _SC[sset]

_MANIFESTS = {
    "paper_test":       ROOT / ".venv/lib/python3.11/site-packages/assaybench/data/"
                               "screen_sets/assayloop-test.yaml",
    "paper_validation": ROOT / ".venv/lib/python3.11/site-packages/assaybench/data/"
                               "screen_sets/assayloop-validation.yaml",
    "validation":       REPRO / "donor" / "validation_fold_218.yaml",
    "test":             REPRO / "donor" / "test_fold_334.yaml",
}

_PUB = {}
def publications_for(sset):
    """{screen: publication_id}. Empty dict means publication clustering is unavailable
    for that set, and the figures fall back to screen-level with a printed warning."""
    if sset in _PUB:
        return _PUB[sset]
    if sset == "bundle18":
        # the reproduction bundle carries publication_id per recorded campaign
        mf = REPRO / "bundle" / "manifest.json"
        out = {}
        if mf.exists():
            for e in json.loads(mf.read_text()):
                out[e["cell"]["screen_id"]] = str(e["publication_id"])
        _PUB[sset] = out
        return out
    p = _MANIFESTS.get(sset)
    out = {}
    if p is not None and Path(p).exists():
        import yaml
        for s in yaml.safe_load(Path(p).read_text())["screens"]:
            out[s["dataset_name"]] = str(s["source_id"])
    _PUB[sset] = out
    return out

_META = {}
def meta_for(sset):
    """{screen: {...}} — library size, hits available, type dimensions, publication."""
    if sset in _META:
        return _META[sset]
    try:
        from assayloop.screen_direction import classify_selection
    except Exception:
        classify_selection = None
    pubs = publications_for(sset)
    short = {"Drug / Chemical / Environmental Response": "Drug / chemical",
             "Fitness / Proliferation / Viability": "Fitness",
             "Host-Pathogen / Infection Response": "Host-pathogen",
             "Molecular Output / Reporter / Pathway Activity": "Molecular output",
             "Trafficking / Localization / Structural Phenotypes": "Trafficking"}
    out = {}
    for s in screens_for(sset):
        phen = str(getattr(s, "cleaned_phenotype", "") or "?").strip()
        direction = "?"
        if classify_selection is not None:
            try:
                ctx = s.context() if hasattr(s, "context") else {}
                direction = classify_selection(
                    f"{ctx.get('phenotype') or ''}. {ctx.get('notes') or ''}")[0].capitalize()
            except Exception:
                pass
        out[s.dataset_name] = {
            "n_genes": len(s.genes), "total_hits": int(s.total_hits),
            "pub": pubs.get(s.dataset_name, f"?{s.dataset_name}"),
            "Perturbation": str(getattr(s, "library_methodology", "") or "?").strip(),
            "Phenotype class": short.get(phen, phen[:22]),
            "Selection direction": direction,
            "Library type": str(getattr(s, "library_type", "") or "?").strip(),
        }
    _META[sset] = out
    return out

DIMS = ["Perturbation", "Phenotype class", "Selection direction", "Library type"]


def load_external(sset, fn, mean_mode):
    p = EXT / fn
    if not p.exists():
        return []
    rows = []
    for r in json.loads(p.read_text()):
        w = ARM_WEIGHTS.get(r["arm"])
        if w is None:
            continue                      # an unmapped arm name is skipped, not guessed
        af, dm, hsa = w
        mm = mean_mode if hsa > 0 else "n/a"
        rows.append(dict(label=canon_label(af, dm, hsa, mm), group=arm_group(af, dm, hsa),
                         af_w=af, dm_w=dm, hsa_w=hsa, mean_mode=mm, source=fn,
                         sset=sset, screen=r["screen"], hits=int(r["hits"]),
                         ef=float(r["ef"]), picks=int(r.get("picks", BUDGET)),
                         n_measured=int(r.get("n_measured", r.get("picks", BUDGET)))))
    return rows


def load_sweep(sset, sid, prefix, af, dm, hsa):
    try:
        base = sweep_base(sid)
    except Exception:
        return []
    rows = []
    for d in sorted(base.glob(f"{sid}-*")):
        rp = d / "result.json"
        if not rp.is_dir() and rp.exists():
            try:
                rd = json.loads(rp.read_text())
            except Exception:
                continue
            fm = rd.get("final_metrics") or {}
            if "n_hits" not in fm:
                continue
            screen = d.name.split("-", 2)[-1] if d.name.count("-") >= 2 else d.name
            # strip the "<sid>-NN-" prefix exactly
            m = re.match(re.escape(sid) + r"-\d+-(.+)$", d.name)
            screen = m.group(1) if m else screen
            lib = set(screens_by_name(sset).get(screen, ()) or ())
            picks, n_meas = 0, 0
            for st in rd.get("steps", []):
                for g in (st.get("acquired_batch") or []):
                    picks += 1
                    if not lib or g in lib:
                        n_meas += 1
            rows.append(dict(
                label=canon_label(af, dm, hsa, None, prefix=prefix),
                group=arm_group(af, dm, hsa, prefix=prefix),
                af_w=af, dm_w=dm, hsa_w=hsa, mean_mode="n/a", source=sid,
                sset=sset, screen=screen, hits=int(fm["n_hits"]),
                ef=float(fm.get("n_hits_vs_random", np.nan)),
                picks=picks or BUDGET, n_measured=n_meas or picks or BUDGET))
    return rows

_BYNAME = {}
def screens_by_name(sset):
    if sset not in _BYNAME:
        _BYNAME[sset] = {s.dataset_name: set(s.genes) for s in screens_for(sset)}
    return _BYNAME[sset]

In [ ]:
# --- build the master table --------------------------------------------------
rows = []
_ext, _swp = active_sources()
for sset, fn, mm in _ext:
    rows += load_external(sset, fn, mm)
for sset, sid, pre, af, dm, hsa in _swp:
    rows += load_sweep(sset, sid, pre, af, dm, hsa)

RAW = pd.DataFrame(rows)
if RAW.empty:
    raise SystemExit("no rows loaded — run verify_sources() and check the paths")

# attach screen metadata from the SCREEN RECORDS, never from the run files, so n_genes /
# total_hits / pub / type are identical across arms by construction
meta_rows = []
for sset in sorted(RAW.sset.unique()):
    M = meta_for(sset)
    for k, v in M.items():
        meta_rows.append(dict(sset=sset, screen=k, **v))
META = pd.DataFrame(meta_rows)
RAW = RAW.drop(columns=[c for c in ("n_genes", "total_hits", "pub") if c in RAW.columns])
RAW = RAW.merge(META, on=["sset", "screen"], how="left")

# arms that recorded the same (label, sset, screen) twice -- e.g. a file re-run -- keep the
# LAST source listed in the registry, and say so rather than silently averaging
dup = RAW.duplicated(subset=["label", "sset", "screen"], keep="last")
if dup.any():
    print(f"dropped {int(dup.sum())} duplicate (arm, screen) rows; later registry entries win")
    print(RAW[dup].groupby(["sset", "label", "source"]).size().to_string())
RAW = RAW[~dup].reset_index(drop=True)

RAW["hit_rate"]   = RAW.hits / RAW.n_measured.replace(0, np.nan)
RAW["rate_slots"] = RAW.hits / RAW.picks.replace(0, np.nan)
RAW["recall"]     = RAW.hits / RAW.total_hits.replace(0, np.nan)
RAW["screen_rate"] = RAW.total_hits / RAW.n_genes.replace(0, np.nan)

METRICS = {"ef": "adjusted EF", "hits": "unique measured hits",
           "hit_rate": "hit rate (hits / measured picks)",
           "rate_slots": "hit rate (hits / slots offered)",
           "recall": "recall (hits / hits available)"}

def apply_filters(df):
    c = _criteria()
    out = df
    if GENOME_SCALE_ONLY:
        out = out[out.n_genes > BUDGET + 1]
    if c["min_genes"]:  out = out[out.n_genes >= c["min_genes"]]
    if c["max_genes"]:  out = out[out.n_genes <= c["max_genes"]]
    if c["min_hits"]:   out = out[out.total_hits >= c["min_hits"]]
    if c["max_rate"]:   out = out[out.screen_rate <= c["max_rate"]]
    return out

LONG = apply_filters(RAW).reset_index(drop=True)

print(f"RAW  {len(RAW):>6} rows   {RAW.label.nunique()} arms   {RAW.sset.nunique()} screen sets")
print(f"LONG {len(LONG):>6} rows after filters "
      f"(GENOME_SCALE_ONLY={GENOME_SCALE_ONLY}, CURATED_CRITERIA={CURATED_CRITERIA})")
print()
print(LONG.groupby("sset").agg(screens=("screen", "nunique"), arms=("label", "nunique"),
                               pubs=("pub", "nunique")).to_string())

## 3. Selecting arms

`pick()` resolves each term in a fixed order — **exact label, then exact group, then a
shorthand, then substring**. The group check comes before the substring check deliberately:
there is a group named `HSA` and a substring `HSA` that matches far more, so without the
ordering `select=["HSA"]` would quietly expand to every HSA-bearing arm.

In [ ]:
# --- selection ---------------------------------------------------------------
SHORTHAND = {
    "all":   lambda L: sorted(L.label.unique()),
    "hsa":   lambda L: sorted(L[L.hsa_w > 0].label.unique()),
    "glm5":  lambda L: sorted(L[L.group == "GLM-5"].label.unique()),
    "ours":  lambda L: sorted(L[L.group != "GLM-5"].label.unique()),
    "noaf":  lambda L: sorted(L[(L.hsa_w > 0) & (L.af_w == 0)].label.unique()),
    "real":  lambda L: sorted(L[L.mean_mode != "zero"].label.unique()),
}

def pick(select=None, df=None, cap=None):
    """Resolve `select` to an ordered, de-duplicated list of canonical labels."""
    L = LONG if df is None else df
    if select is None:
        select = DEFAULT_SELECT
    if select is None:
        select = [STANDARD] if STANDARD in set(L.label) else SHORTHAND["all"](L)
    if isinstance(select, str):
        select = [select]
    labels, groups = set(L.label), set(L.group)
    out = []
    for term in select:
        if term in labels:                       # 1. exact label
            hits = [term]
        elif term in groups:                     # 2. exact GROUP -- before substring
            hits = sorted(L[L.group == term].label.unique())
        elif term.lower() in SHORTHAND:          # 3. shorthand
            hits = SHORTHAND[term.lower()](L)
        else:                                    # 4. substring, case-insensitive
            hits = sorted(l for l in labels if term.lower() in l.lower())
        if not hits:
            print(f"!! nothing matched {term!r}")
        for h in hits:
            if h not in out:
                out.append(h)
    cap = MAX_ARMS_PER_FIGURE if cap is None else cap
    if cap and len(out) > cap:
        print(f"!! {len(out)} arms selected, drawing the first {cap} "
              f"(the palette holds {cap}); narrow `select=` to choose")
        out = out[:cap]
    return out


def list_arms(sset=None, metric="ef"):
    """Every arm, as a copy-pasteable `select=` block. Highest `metric` first."""
    L = LONG if sset is None else LONG[LONG.sset == sset]
    if L.empty:
        print("no rows"); return
    agg = (L.groupby(["label", "group"])
             .agg(v=(metric, "mean"), n=("screen", "nunique"),
                  ssets=("sset", "nunique"))
             .reset_index().sort_values("v", ascending=False))
    w = max(len(x) for x in agg.label) + 2
    print(f"# {metric} mean, {'one screen set: ' + sset if sset else 'all screen sets'}")
    print("SELECT = [")
    for r in agg.itertuples():
        print(f'    {r.label!r:<{w}}# {r.v:>8.3f}  {r.group:<11} '
              f'n={r.n:<4} sets={r.ssets}')
    print("]")

list_arms()

## 4. Aggregation

`agg_metric` is the one place that decides how per-screen numbers become one number, and it
honours `CLUSTER_BY`. `hits` always aggregates as a **sum** (a count), everything else as a
**mean**, because a mean of hit counts hides how much was available.

`boot_delta` bootstraps the paired difference between two arms over whichever unit
`CLUSTER_BY` names. Every CI in this notebook is paired and over clusters, never over rows.

In [ ]:
# --- aggregation -------------------------------------------------------------
def _unit(df):
    if CLUSTER_BY == "publication":
        if df.pub.astype(str).str.startswith("?").all():
            print("!! no publication map for this set — falling back to screen-level")
            return "screen"
        return "pub"
    return "screen"

def agg_metric(df, metric):
    """One number for a frame. `hits` sums; everything else means over the cluster unit."""
    if df.empty:
        return np.nan
    if metric == "hits":
        return float(df.hits.sum())
    u = _unit(df)
    if u == "screen":
        return float(df[metric].mean())
    return float(df.groupby("pub")[metric].mean().mean())

def arm_frame(label, sset, df=None):
    L = LONG if df is None else df
    return L[(L.label == label) & (L.sset == sset)]

def boot_delta(label, base, sset, metric="ef"):
    """Paired delta vs `base`, bootstrapped over clusters. Returns (mean, lo, hi, n, nclus)."""
    a, b = arm_frame(label, sset), arm_frame(base, sset)
    j = a.merge(b, on=["sset", "screen"], suffixes=("", "_b"))
    if j.empty:
        return (np.nan,) * 3 + (0, 0)
    d = j[metric] - j[f"{metric}_b"]
    # for `hits` (a count) the BAR is the total difference while the interval is on the
    # per-cluster mean, so the two are deliberately on different scales
    u = _unit(j)
    if u == "pub":
        per = j.assign(d=d).groupby("pub")["d"].mean().to_numpy()
    else:
        per = d.to_numpy()
    rng = np.random.default_rng(RNG_SEED)
    if len(per) < 2:
        return (float(per.mean()), np.nan, np.nan, len(j), len(per))
    bs = per[rng.integers(0, len(per), (N_BOOT, len(per)))].mean(1)
    m = float(d.sum()) if metric == "hits" else float(per.mean())
    return (m, float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)),
            len(j), len(per))

def summary_table(select=None, sset=None, metric="ef", base=None):
    """The table behind the figures. `base=None` picks AF alone when present."""
    ssets = [sset] if sset else sorted(LONG.sset.unique())
    labels = pick(select)
    rows = []
    for st in ssets:
        present = [l for l in labels if not arm_frame(l, st).empty]
        bse = base or ("AF alone" if "AF alone" in present else (present[0] if present else None))
        for l in present:
            f = arm_frame(l, st)
            d = boot_delta(l, bse, st, metric) if bse and l != bse else (0.0, np.nan, np.nan,
                                                                         f.screen.nunique(), 0)
            rows.append(dict(sset=st, label=l, group=f.group.iloc[0],
                             af_w=f.af_w.iloc[0], dm_w=f.dm_w.iloc[0], hsa_w=f.hsa_w.iloc[0],
                             mean_mode=f.mean_mode.iloc[0],
                             n=f.screen.nunique(), pubs=f.pub.nunique(),
                             value=agg_metric(f, metric),
                             median=float(f[metric].median()) if metric != "hits" else np.nan,
                             hits=int(f.hits.sum()),
                             avail=int(f.groupby("screen").total_hits.first().sum()),
                             base=bse, delta=d[0], lo=d[1], hi=d[2]))
    T = pd.DataFrame(rows)
    if not T.empty:
        T["pct_avail"] = 100 * T.hits / T.avail.replace(0, np.nan)
        T["sig"] = np.where(T.lo.notna() & ((T.lo > 0) | (T.hi < 0)), "*", "")
    return T

## Figure 1 — any metric, any arms, faceted by screen set

`metric=` picks the quantity, `select=` the arms. The standard method gets a heavy outline.
Bars sort descending, so the legend sits **below** the panels — an in-axes legend would cover
the value labels on the best arms.

In [ ]:
# --- Figure 1: absolute metric by arm ----------------------------------------
def fig_absolute(metric="ef", select=None, ssets=None, annotate=True, figsize=None):
    T = summary_table(select, metric=metric)
    if T.empty:
        print("nothing to draw"); return
    ssets = ssets or sorted(T.sset.unique())
    T = T[T.sset.isin(ssets)]
    labels = [l for l in pick(select) if l in set(T.label)]
    cmap = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(labels)}
    n = len(ssets)
    fig, axes = plt.subplots(1, n, figsize=figsize or (4.6 * n, 0.42 * len(labels) + 2.4),
                             squeeze=False)
    for ax, st in zip(axes[0], ssets):
        S = T[T.sset == st].set_index("label").reindex(labels).dropna(subset=["value"])
        S = S.sort_values("value")
        y = np.arange(len(S))
        ax.barh(y, S.value, color=[cmap[i] for i in S.index],
                edgecolor=["black" if i == STANDARD else "none" for i in S.index],
                linewidth=[1.8 if i == STANDARD else 0 for i in S.index])
        ax.set_yticks(y); ax.set_yticklabels([])
        ax.set_title(f"{st}  (n={int(S.n.max()) if len(S) else 0}"
                     f"{', ' + str(int(S.pubs.max())) + ' pubs' if CLUSTER_BY=='publication' and len(S) else ''})",
                     fontsize=9)
        ax.set_xlabel(METRICS.get(metric, metric))
        if annotate:
            for yi, (v, lab) in enumerate(zip(S.value, S.index)):
                ax.text(v, yi, f" {v:,.3f}" if metric != "hits" else f" {v:,.0f}",
                        va="center", fontsize=7.5,
                        bbox=dict(fc="white", ec="none", alpha=0.75, pad=0.8))
    handles = [mpl.patches.Patch(color=cmap[l], label=l) for l in labels]
    fig.legend(handles=handles, loc="lower center", ncol=min(3, len(labels)),
               frameon=False, fontsize=8)
    fig.suptitle(f"{METRICS.get(metric, metric)}  —  "
                 f"{'publication-mean' if CLUSTER_BY=='publication' else 'screen-mean'}"
                 f"{'  (sum)' if metric=='hits' else ''}", y=1.0, fontsize=10)
    fig.tight_layout(rect=(0, 0.06 + 0.02 * (len(labels) // 3), 1, 0.97))
    plt.show()
    return T

fig_absolute("ef");

In [ ]:
fig_absolute("hits");

## Figure 2 — paired change vs a baseline, with clustered CIs

`base=` chooses the reference (default `AF alone`). A star marks an interval excluding zero.
For `metric="hits"` the bar is the **total** difference and the interval is on the per-cluster
mean, which is why the two can disagree in scale.

In [ ]:
# --- Figure 2: paired delta --------------------------------------------------
def fig_delta(metric="ef", select=None, base=None, ssets=None, figsize=None):
    T = summary_table(select, metric=metric, base=base)
    T = T[T.label != T.base]
    if T.empty:
        print("nothing to draw"); return
    ssets = ssets or sorted(T.sset.unique())
    T = T[T.sset.isin(ssets)]
    labels = [l for l in pick(select) if l in set(T.label)]
    n = len(ssets)
    fig, axes = plt.subplots(1, n, figsize=figsize or (4.6 * n, 0.42 * len(labels) + 2.4),
                             sharex=False, squeeze=False)
    for ax, st in zip(axes[0], ssets):
        S = T[T.sset == st].set_index("label").reindex(labels).dropna(subset=["delta"])
        S = S.sort_values("delta")
        y = np.arange(len(S))
        col = ["#55A868" if v > 0 else "#C44E52" for v in S.delta]
        ax.barh(y, S.delta, color=col, alpha=0.85)
        if S.lo.notna().any():
            ax.errorbar(S.delta, y, xerr=[S.delta - S.lo, S.hi - S.delta],
                        fmt="none", ecolor="black", elinewidth=0.9, capsize=2.5)
        ax.axvline(0, color="black", lw=1)
        ax.set_yticks(y)
        ax.set_yticklabels([f"{i}{'*' if s=='*' else ''}" for i, s in zip(S.index, S.sig)]
                           if ax is axes[0][0] else [], fontsize=7.5)
        ax.set_title(f"{st}  vs {S.base.iloc[0] if len(S) else base}", fontsize=9)
        ax.set_xlabel(f"change in {METRICS.get(metric, metric)}")
    fig.suptitle(f"paired change vs baseline — {CLUSTER_BY}-clustered 95% CI, "
                 f"* excludes zero", y=1.0, fontsize=10)
    fig.tight_layout(rect=(0, 0.02, 1, 0.96))
    plt.show()
    return T

fig_delta("ef");

## Figure 3 — the RRF weight sweep

The axis the DepMap weight actually moves along. By default it draws the no-AF-term family
(`af_w == 0, hsa_w == 1`), where `dm_w` is the only thing varying, and puts `AF + DepMap` on
as a horizontal reference. **The optimum moves between screen sets** — 0.5 on `paper_test`,
0.25 on `paper_validation` — so read one panel at a time.

In [ ]:
# --- Figure 3: metric vs DepMap weight ---------------------------------------
def fig_weight_sweep(metric="ef", family="noaf", ssets=None, refs=("AF + DepMap (dm=0.5)",
                                                                   "AF alone"),
                     mean_mode=None, figsize=None):
    """family: 'noaf' (af=0, hsa=1) | 'afhsa' (af=1, hsa>0) | 'nohsa' (hsa=0)."""
    L = LONG
    if family == "noaf":
        F = L[(L.af_w == 0) & (L.hsa_w == 1)]
    elif family == "afhsa":
        F = L[(L.af_w > 0) & (L.hsa_w > 0)]
    else:
        F = L[(L.hsa_w == 0) & (L.dm_w >= 0)]
    if mean_mode:
        F = F[F.mean_mode.isin([mean_mode, "n/a"])]
    if F.empty:
        print("no arms in that family"); return
    ssets = ssets or sorted(F.sset.unique())
    modes = [m for m in ("real", "zero", "n/a") if m in set(F.mean_mode)]
    n = len(ssets)
    fig, axes = plt.subplots(1, n, figsize=figsize or (4.2 * n, 3.4), squeeze=False)
    for ax, st in zip(axes[0], ssets):
        for mi, mm in enumerate(modes):
            G = F[(F.sset == st) & (F.mean_mode == mm)]
            if G.empty:
                continue
            pts = sorted({w for w in G.dm_w})
            vals = [agg_metric(G[G.dm_w == w], metric) for w in pts]
            ax.plot(pts, vals, "o-", color=PALETTE[mi], lw=1.6, ms=5,
                    label=f"DepMap + HSA [{'real mean' if mm=='real' else 'mean=0'}]"
                          if mm != "n/a" else "DepMap + HSA")
            for w, v in zip(pts, vals):
                ax.annotate(f"{v:,.2f}" if metric != "hits" else f"{v:,.0f}", (w, v),
                            textcoords="offset points", xytext=(0, 7), ha="center",
                            fontsize=7)
        for ri, ref in enumerate(refs or ()):
            R = arm_frame(ref, st)
            if R.empty:
                continue
            ax.axhline(agg_metric(R, metric), color=NEUTRAL, ls=(0, (4, 2)), lw=1.3)
            ax.annotate(ref, (ax.get_xlim()[0], agg_metric(R, metric)), fontsize=7,
                        color="#666", va="bottom", xytext=(2, 2),
                        textcoords="offset points")
        ax.set_title(st, fontsize=9)
        ax.set_xlabel("DepMap weight in the RRF sum  (hsa = 1, af = 0)")
        ax.set_ylabel(METRICS.get(metric, metric))
    h, l = axes[0][0].get_legend_handles_labels()
    if h:
        fig.legend(h, l, loc="lower center", ncol=min(3, len(l)), frameon=False, fontsize=8)
    fig.suptitle(f"{METRICS.get(metric, metric)} against the DepMap RRF weight", y=1.0,
                 fontsize=10)
    fig.tight_layout(rect=(0, 0.12, 1, 0.95))
    plt.show()

fig_weight_sweep("ef")

In [ ]:
fig_weight_sweep("hits")

## Figures 4-5 — performance by screen type

`dim=` picks the cut: `Perturbation` (knockout / CRISPRi / CRISPRa — the cut that reproduces
the direction-matching result), `Phenotype class`, `Selection direction`, `Library type`.
Categories below `MIN_N` screens are dropped rather than drawn with a CI wider than the
effect.

In [ ]:
# --- Figures 4-5: by screen type ---------------------------------------------
MIN_N = 5

def type_table(metric="ef", select=None, dim="Perturbation", sset=None, base=None,
               ci=True):
    """Per-category value for every selected arm. `ci=False` skips the bootstrap,
    which an absolute figure does not need and which costs N_BOOT draws per
    (category x arm)."""
    ssets = [sset] if sset else sorted(LONG.sset.unique())
    labels = pick(select)
    rows = []
    for st in ssets:
        present = [l for l in labels if not arm_frame(l, st).empty]
        bse = base or ("AF alone" if "AF alone" in present else
                       (present[0] if present else None))
        for cat in sorted(LONG[LONG.sset == st][dim].dropna().unique()):
            for l in present:
                f = arm_frame(l, st)
                f = f[f[dim] == cat]
                if f.screen.nunique() < MIN_N:
                    continue
                d = (np.nan,) * 5
                if ci and bse and l != bse:
                    a, b = f, arm_frame(bse, st)
                    j = a.merge(b, on=["sset", "screen"], suffixes=("", "_b"))
                    if not j.empty:
                        diff = j[metric] - j[f"{metric}_b"]
                        per = (j.assign(d=diff).groupby("pub")["d"].mean().to_numpy()
                               if _unit(j) == "pub" else diff.to_numpy())
                        rng = np.random.default_rng(RNG_SEED)
                        if len(per) >= 2:
                            bs = per[rng.integers(0, len(per), (N_BOOT, len(per)))].mean(1)
                            d = (float(per.mean()), float(np.percentile(bs, 2.5)),
                                 float(np.percentile(bs, 97.5)), len(j), len(per))
                        else:
                            d = (float(per.mean()), np.nan, np.nan, len(j), len(per))
                rows.append(dict(sset=st, dim=dim, cat=cat, label=l, base=bse,
                                 n=f.screen.nunique(), value=agg_metric(f, metric),
                                 hits=int(f.hits.sum()),
                                 delta=d[0], lo=d[1], hi=d[2]))
    return pd.DataFrame(rows)


def fig_by_type(metric="ef", select=None, dim="Perturbation", mode="absolute",
                ssets=None, figsize=None):
    """mode: 'absolute' | 'delta'."""
    T = type_table(metric, select, dim)
    if T.empty:
        print("nothing to draw"); return
    ssets = ssets or sorted(T.sset.unique())
    T = T[T.sset.isin(ssets)]
    labels = [l for l in pick(select) if l in set(T.label)]
    cats = sorted(T.cat.unique())
    cmap = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(labels)}
    n = len(ssets)
    fig, axes = plt.subplots(1, n, figsize=figsize or (max(4.6, 1.1 * len(cats)) * n,
                                                       3.8), squeeze=False)
    col = "value" if mode == "absolute" else "delta"
    for ax, st in zip(axes[0], ssets):
        S = T[T.sset == st]
        x = np.arange(len(cats)); w = 0.8 / max(1, len(labels))
        for i, l in enumerate(labels):
            vals, los, his = [], [], []
            for c in cats:
                r = S[(S.cat == c) & (S.label == l)]
                vals.append(r[col].iloc[0] if len(r) else np.nan)
                los.append(r.lo.iloc[0] if len(r) and mode == "delta" else np.nan)
                his.append(r.hi.iloc[0] if len(r) and mode == "delta" else np.nan)
            pos = x - 0.4 + w * (i + 0.5)
            ax.bar(pos, vals, width=w * 0.92, color=cmap[l],
                   edgecolor="black" if l == STANDARD else "none",
                   linewidth=1.4 if l == STANDARD else 0)
            if mode == "delta" and np.isfinite(los).any():
                lo_err = [v - lo if np.isfinite(lo) else 0 for v, lo in zip(vals, los)]
                hi_err = [hi - v if np.isfinite(hi) else 0 for v, hi in zip(vals, his)]
                ax.errorbar(pos, vals, yerr=[lo_err, hi_err], fmt="none",
                            ecolor="black", elinewidth=0.8, capsize=2)
        if mode == "delta":
            ax.axhline(0, color="black", lw=1)
        ns = [int(S[S.cat == c].n.max()) if len(S[S.cat == c]) else 0 for c in cats]
        ax.set_xticks(x)
        ax.set_xticklabels([f"{c}\nn={k}" for c, k in zip(cats, ns)], fontsize=7.5)
        ax.set_title(st, fontsize=9)
        ax.set_ylabel((("change in " if mode == "delta" else "")
                       + METRICS.get(metric, metric)))
    handles = [mpl.patches.Patch(color=cmap[l], label=l) for l in labels]
    fig.legend(handles=handles, loc="lower center", ncol=min(3, len(labels)),
               frameon=False, fontsize=8)
    fig.suptitle(f"{METRICS.get(metric, metric)} by {dim}"
                 f"{' — change vs baseline' if mode=='delta' else ''}", y=1.0, fontsize=10)
    fig.tight_layout(rect=(0, 0.08 + 0.03 * (len(labels) // 3), 1, 0.95))
    plt.show()
    return T

fig_by_type("ef", dim="Perturbation", mode="absolute");

In [ ]:
fig_by_type("ef", dim="Phenotype class", mode="delta");

## Figure 6 — the two metrics disagree, and this shows it

EF against hits, one point per arm. **The whole HSA-vs-AF+DepMap story lives in this panel:**
the HSA arms sit to the right (more hits) and below (less EF). If a future arm lands up and to
the right of everything, that is a real improvement rather than a metric trade.

Why it happens: `EF = h / (b_eff * H/D) = precision / prevalence`, so one hit on a screen with
`H/D = 0.05%` is worth ~19 EF while one hit at 8% is worth ~0.14. Arms that gain hits on dense
screens and lose a handful on near-empty ones move right and down.

In [ ]:
# --- Figure 6: EF against hits -----------------------------------------------
def fig_metric_tradeoff(select=None, x="hits", y="ef", ssets=None, figsize=None):
    labels = pick(select)
    ssets = ssets or sorted(LONG.sset.unique())
    n = len(ssets)
    fig, axes = plt.subplots(1, n, figsize=figsize or (4.2 * n, 3.6), squeeze=False)
    for ax, st in zip(axes[0], ssets):
        for i, l in enumerate(labels):
            f = arm_frame(l, st)
            if f.empty:
                continue
            ax.scatter(agg_metric(f, x), agg_metric(f, y), s=90,
                       color=PALETTE[i % len(PALETTE)],
                       edgecolor="black" if l == STANDARD else "white",
                       linewidth=1.6 if l == STANDARD else 0.8, zorder=3, label=l)
        ax.set_xlabel(METRICS.get(x, x)); ax.set_ylabel(METRICS.get(y, y))
        ax.set_title(st, fontsize=9)
    handles, lab = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, lab, loc="lower center", ncol=min(3, len(lab)), frameon=False,
               fontsize=8)
    fig.suptitle("the two metrics against each other — up and to the RIGHT is a real win",
                 y=1.0, fontsize=10)
    fig.tight_layout(rect=(0, 0.08 + 0.03 * (len(lab) // 3), 1, 0.95))
    plt.show()

fig_metric_tradeoff()

## Figure 7 — per-screen, two arms head to head

Where a difference actually comes from. Points above the diagonal favour `b`. Marker area
scales with hits available, which is what makes the leverage visible: the biggest EF swings
sit on the smallest markers.

In [ ]:
# --- Figure 7: per-screen scatter --------------------------------------------
def fig_screen_scatter(a, b, metric="ef", sset=None, figsize=(4.6, 4.4), label_top=6):
    sset = sset or sorted(LONG.sset.unique())[0]
    A, B = arm_frame(a, sset), arm_frame(b, sset)
    j = A.merge(B, on=["sset", "screen"], suffixes=("_a", "_b"))
    if j.empty:
        print(f"no shared screens for {a!r} and {b!r} on {sset}"); return
    fig, ax = plt.subplots(figsize=figsize)
    s = 12 + 90 * (j.total_hits_a / max(1, j.total_hits_a.max())) ** 0.5
    d = j[f"{metric}_b"] - j[f"{metric}_a"]
    ax.scatter(j[f"{metric}_a"], j[f"{metric}_b"], s=s, alpha=0.75,
               c=["#55A868" if v > 0 else "#C44E52" for v in d],
               edgecolor="white", linewidth=0.5, zorder=3)
    lim = [0, float(np.nanmax([j[f"{metric}_a"].max(), j[f"{metric}_b"].max()])) * 1.05]
    ax.plot(lim, lim, color="black", lw=1, ls=(0, (4, 2)))
    ax.set_xlim(lim); ax.set_ylim(lim)
    for r in j.reindex(d.abs().sort_values(ascending=False).index[:label_top]).itertuples():
        ax.annotate(r.screen, (getattr(r, f"{metric}_a"), getattr(r, f"{metric}_b")),
                    fontsize=7, xytext=(4, 3), textcoords="offset points")
    ax.set_xlabel(f"{a}\n{METRICS.get(metric, metric)}")
    ax.set_ylabel(f"{b}\n{METRICS.get(metric, metric)}")
    up = int((d > 0).sum()); dn = int((d < 0).sum())
    ax.set_title(f"{sset} — {up} up / {dn} down, marker area = hits available",
                 fontsize=9)
    fig.tight_layout(); plt.show()
    return j

fig_screen_scatter("AF + DepMap (dm=0.5)", "DepMap + HSA (dm=0.25) [real mean]",
                   sset="validation");

## Figure 8 — where the undiscovered hits are, by screen family

Figures 4-5 ask how *well* each method does inside a screen category. This asks what is
**left** — the question for deciding where more effort would pay.

**One split makes it honest.** Many screens hold more hits than the campaign has picks, so a
naive `total_hits - found` counts hits no policy could have reached. Every screen is split:

| | |
|---|---|
| `found` | hits the arm actually acquired |
| `reachable` | `min(total_hits, 1000)` — the most **any** policy could find here |
| `unreachable` | `max(0, total_hits - 1000)` — beyond the budget, not headroom at all |

**The bars plot `reachable - found`** — what a better method could genuinely still win. The
`unreachable` column stays in the table so it is never quietly dropped.

`metric=` switches the question:

| metric | question |
|---|---|
| `missed` | how many reachable hits are still unfound (mean per screen) — absolute opportunity |
| `share` | the same as a % of what was reachable — comparable across categories whose screens differ wildly in size |
| `recall` | the flip side: % of reachable hits actually found |

`dim=` takes any of `Perturbation`, `Phenotype class`, `Selection direction`, `Library type`.
Categories below `MIN_N` screens are dropped, same floor as Figures 4-5.

**This figure is scoped to the full 218-screen validation fold** (`FAMILY_SETS`), unlike the
others, which facet across every loaded screen set. Headroom is a question about a
representative panel, and the curated 20-screen sets are neither representative — a Gemini
signal floor and `num_hits_min: 50` were applied when they were selected — nor large enough to
cut four ways.

**To add the 334-screen test fold later:** set `INCLUDE_TEST_FOLD = True` in the registry cell,
re-run it and the master-table cell, then `FAMILY_SETS = ["validation", "test"]`. That gives
`AF alone` and `AF + DepMap` on both folds immediately — both sweeps already exist at 334
screens. HSA on the test fold does not exist yet, and it is the one piece that costs anything:
those screen descriptions are **not** in the shipped embedding cache, so it needs
`OPENAI_API_KEY` and ~314 embeddings. Everything else there is free — banks rebuild from public
data and the runs make no provider calls.

In [ ]:
# --- Figure 8: headroom by screen family -------------------------------------
# A screen's answer key often holds more hits than the campaign has picks, so the naive
# `total_hits - found` counts hits NO policy could have reached and overstates headroom.
# Hence the reachable / unreachable split below. `reachable = min(total_hits, BUDGET)`.
# WHICH SCREEN SETS THIS FIGURE USES. Scoped to the full 218-screen validation FOLD,
# because headroom is a question about a representative panel and the curated 20-screen
# sets are neither representative (a Gemini signal floor and num_hits_min: 50 were applied)
# nor large enough to cut four ways. To add the 334-screen test fold later:
#
#   1. set INCLUDE_TEST_FOLD = True in the registry cell and re-run it + the master table
#   2. FAMILY_SETS = ["validation", "test"]
#
# That alone gives AF alone and AF + DepMap on both folds -- both sweeps already exist.
# HSA on the test fold does NOT exist yet and is the one piece that costs something: its
# screen descriptions are not in the shipped embedding cache, so it needs OPENAI_API_KEY
# and ~314 embeddings. ASK FIRST -- that is a billed call. Check with
#   python scripts/diagnostics/embed_screen_set.py --screen-set test --dry-run
# Everything else is free: banks rebuild from public data
# (scripts/external/rebuild_banks.py, HANDOFF 24.20) and the runs make no provider calls.
FAMILY_SETS = ["validation"]

HEADROOM_LAB = {
    "missed": "reachable hits still not found (mean per screen)",
    "share":  "share of reachable hits still not found (%)",
    "recall": "share of reachable hits found (%)",
}

def headroom_table(select=None, dim="Perturbation", sset=None):
    """Per-category headroom for every selected arm. Honours CLUSTER_BY."""
    avail = set(LONG.sset.unique())
    ssets = ([sset] if sset else [s for s in FAMILY_SETS if s in avail])
    if not ssets:
        print(f"!! none of FAMILY_SETS={FAMILY_SETS} is loaded; "
              f"available: {sorted(avail)}")
        return pd.DataFrame()
    labels = pick(select)
    rows = []
    for st in ssets:
        for l in [x for x in labels if not arm_frame(x, st).empty]:
            f = arm_frame(l, st)
            for cat in sorted(LONG[LONG.sset == st][dim].dropna().unique()):
                g = f[f[dim] == cat]
                if g.screen.nunique() < MIN_N:
                    continue
                g = g.assign(
                    reachable=np.minimum(g.total_hits, BUDGET),
                    unreachable=np.maximum(0, g.total_hits - BUDGET))
                g = g.assign(missed=g.reachable - g.hits)
                # per-screen quantities first, then averaged over the CLUSTER_BY unit so
                # one prolific publication cannot dominate a category
                if _unit(g) == "pub":
                    p = g.groupby("pub")[["hits", "reachable", "unreachable", "missed",
                                          "total_hits"]].mean()
                else:
                    p = g[["hits", "reachable", "unreachable", "missed", "total_hits"]]
                found, rch = float(p.hits.mean()), float(p.reachable.mean())
                rows.append(dict(
                    sset=st, dim=dim, cat=cat, label=l, group=g.group.iloc[0],
                    n=g.screen.nunique(), pubs=g.pub.nunique(),
                    found=found, reachable=rch,
                    unreachable=float(p.unreachable.mean()),
                    total=float(p.total_hits.mean()),
                    missed=float(p.missed.mean()),
                    share=100 * float(p.missed.mean()) / rch if rch else np.nan,
                    recall=100 * found / rch if rch else np.nan))
    return pd.DataFrame(rows)


def fig_headroom(metric="missed", select=None, dim="Perturbation", ssets=None,
                 cats=None, figsize=None):
    if metric not in HEADROOM_LAB:
        print(f"metric must be one of {list(HEADROOM_LAB)}"); return
    T = headroom_table(select, dim)
    if T.empty:
        print("nothing to draw"); return
    ssets = ssets or [s for s in FAMILY_SETS if s in set(T.sset)]
    T = T[T.sset.isin(ssets)]
    labels = [l for l in pick(select) if l in set(T.label)]
    cmap = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(labels)}
    n = len(ssets)
    # one row of categories per screen set; ascending so the biggest gap sits at the TOP
    heights = []
    for st in ssets:
        c = sorted(T[T.sset == st].cat.unique())
        heights.append(len([x for x in c if not cats or x in cats]))
    fig, axes = plt.subplots(1, n, squeeze=False,
                             figsize=figsize or (5.8 * n,
                                                 0.46 * max(heights) * max(1, len(labels))
                                                 + 2.0))
    for ax, st in zip(axes[0], ssets):
        S = T[T.sset == st]
        order = (S[S.label == labels[0]].sort_values(metric, ascending=(metric != "recall"))
                 .cat.tolist()) if labels else []
        order = [c for c in order if not cats or c in cats]
        if not order:
            ax.set_axis_off(); continue
        y = np.arange(len(order)); h = 0.8 / max(1, len(labels))
        for k, l in enumerate(labels):
            ti = S[S.label == l].set_index("cat")
            vals = [ti[metric].get(c, np.nan) for c in order]
            ax.barh(y - 0.4 + h * (k + 0.5), vals, height=h * 0.92, color=cmap[l],
                    edgecolor="black" if l == STANDARD else "none",
                    linewidth=1.6 if l == STANDARD else 0,
                    zorder=5 if l == STANDARD else 3)
        b0 = S[S.label == labels[0]].set_index("cat")
        for yy, c in zip(y, order):
            vals = [S[S.label == l].set_index("cat")[metric].get(c, np.nan) for l in labels]
            note = (f"n={int(b0.n.get(c, 0))}" if metric != "missed" else
                    f"n={int(b0.n.get(c, 0))}  of {b0.reachable.get(c, float('nan')):.0f} reachable")
            ax.text(np.nanmax(vals) * 1.02, yy, note, va="center", fontsize=7,
                    color="#555")
        ax.set_yticks(y)
        ax.set_yticklabels(order if ax is axes[0][0] else [], fontsize=8)
        if metric in ("share", "recall"):
            ax.set_xlim(0, 100)
        else:
            ax.set_xlim(0, float(np.nanmax(S[metric])) * 1.45)
        ax.set_title(f"{dim} — {st}", loc="left", fontsize=9.5)
        ax.set_xlabel(HEADROOM_LAB[metric])
    handles = [mpl.patches.Patch(color=cmap[l], label=l) for l in labels]
    fig.legend(handles=handles, loc="lower center", ncol=min(2, len(labels)),
               frameon=False, fontsize=8)
    fig.suptitle("What is still out there: hits inside the budget that nobody found\n"
                 "longer bar = more undiscovered hits a better method could still win"
                 if metric == "missed" else
                 f"{HEADROOM_LAB[metric]} by {dim}", y=1.0, fontsize=10)
    fig.tight_layout(rect=(0, 0.06 + 0.035 * len(labels), 1, 0.94))
    plt.show()
    return T

HEAD = fig_headroom("missed", dim="Perturbation")

In [ ]:
# the same question as a rate, which is what makes categories of very different size
# comparable -- and the table, where `unreachable` is visible rather than dropped
fig_headroom("share", dim="Phenotype class");

_show = (headroom_table(dim="Phenotype class")
         .sort_values(["sset", "cat", "label"])
         [["sset", "cat", "label", "n", "pubs", "found", "reachable", "missed",
           "recall", "unreachable", "total"]].round(1))
print("`recall` is of what was REACHABLE (min(total_hits, 1000)), not of the whole answer "
      "key.\n`unreachable` hits lie beyond the 1000-pick budget and are headroom for "
      "nobody.")
_show

## Figure 9 — absolute metric within each screen family

The same layout as Figure 8, showing **where each method lands** rather than what is left. One
panel row per screen-type dimension, categories ordered by the first arm's value so a category
sits at the same height in every panel, and **`AF alone` drawn automatically as the reference**
whether or not it is in `select=`.

`metric=` takes any of `ef`, `hits`, `hit_rate`, `rate_slots`, `recall`. `dims=` narrows which
cuts to draw (name or substring), `cats=` which categories (rows) to keep —
`cats=["Knockout", "Activation"]` works. Scoped to `FAMILY_SETS`, the same knob Figure 8 uses.

`mode="delta"` switches to the paired change against the reference with cluster-bootstrapped
intervals, which is the honest form when comparing arms rather than reading levels. Absolute
mode skips the bootstrap entirely (`ci=False`), so it is fast.

In [ ]:
# --- Figure 9: absolute metric by screen family ------------------------------
# Figure 8 asks what is LEFT; this asks where each method LANDS. Same horizontal layout so
# the two read alike: one panel row per dimension, one column per screen set in FAMILY_SETS.
#
# Category order comes from the FIRST selected arm and every arm is drawn in that order, so
# a category sits at the same height across panels. `AF alone` is prepended as the reference
# unless it is already selected, matching compare_all_results.ipynb Figure 8.
def fig_type_metric(metric="ef", select=None, dims=None, cats=None, mode="absolute",
                    reference="AF alone", ssets=None, figsize=None):
    if metric not in METRICS:
        print(f"metric must be one of {list(METRICS)}"); return
    use = [d for d in DIMS if not dims or any(
        str(t).lower() in d.lower() for t in ([dims] if isinstance(dims, str) else dims))]
    if not use:
        print(f"!! no dimension matched {dims!r}; available: {DIMS}"); return
    avail = set(LONG.sset.unique())
    ssets = ssets or [s for s in FAMILY_SETS if s in avail]
    if not ssets:
        print(f"!! none of FAMILY_SETS={FAMILY_SETS} is loaded; available: {sorted(avail)}")
        return
    labels = pick(select)
    if reference and reference in set(LONG.label) and reference not in labels:
        labels = [reference] + labels
    cmap = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(labels)}
    col = "value" if mode == "absolute" else "delta"

    nrow, ncol = len(use), len(ssets)
    fig, axes = plt.subplots(nrow, ncol, squeeze=False,
                             figsize=figsize or (5.6 * ncol,
                                                 (1.6 + 0.30 * len(labels)) * nrow + 1.6))
    seen, kept = {}, []
    for i, dim in enumerate(use):
        for j, sset in enumerate(ssets):
            ax = axes[i][j]
            T = type_table(metric, labels, dim, sset=sset, base=reference,
                           ci=(mode == "delta"))
            if T.empty:
                ax.set_axis_off(); continue
            kept.append(T.assign(metric=metric, mode=mode))
            here = [l for l in labels if l in set(T.label)]
            if mode == "delta":
                here = [l for l in here if l != reference]
            if not here:
                ax.set_axis_off(); continue
            order = T[T.label == here[0]].sort_values(col).cat.tolist()
            if cats:
                want = [cats] if isinstance(cats, str) else list(cats)
                order = [c for c in order
                         if any(str(w).lower() in str(c).lower() for w in want)]
                if not order:
                    print(f"!! no category matched {cats!r} on {dim} / {sset}")
            if not order:
                ax.set_axis_off(); continue
            y = np.arange(len(order)); h = 0.8 / max(1, len(here))
            for k, l in enumerate(here):
                ti = T[T.label == l].set_index("cat")
                vals = [ti[col].get(c, np.nan) for c in order]
                ax.barh(y - 0.4 + h * (k + 0.5), vals, height=h * 0.92, color=cmap[l],
                        edgecolor="black" if l == STANDARD else "white",
                        linewidth=1.6 if l == STANDARD else 0.6,
                        zorder=5 if l == STANDARD else 3, label=l)
                seen[l] = cmap[l]
                if mode == "delta" and ti.lo.notna().any():
                    lo = [ti.lo.get(c, np.nan) for c in order]
                    hi = [ti.hi.get(c, np.nan) for c in order]
                    err = [[v - a if np.isfinite(a) else 0 for v, a in zip(vals, lo)],
                           [b - v if np.isfinite(b) else 0 for v, b in zip(vals, hi)]]
                    ax.errorbar(vals, y - 0.4 + h * (k + 0.5), xerr=err, fmt="none",
                                ecolor="black", elinewidth=0.8, capsize=2, zorder=6)
            b0 = T[T.label == here[0]].set_index("cat")
            for yy, c in zip(y, order):
                vals = [T[T.label == l].set_index("cat")[col].get(c, np.nan) for l in here]
                if np.isfinite(vals).any():
                    ax.text(np.nanmax(vals) * 1.02, yy, f"n={int(b0.n.get(c, 0))}",
                            va="center", fontsize=7, color="#555")
            if mode == "delta":
                ax.axvline(0, color="black", lw=1, zorder=4)
            ax.set_yticks(y)
            ax.set_yticklabels(order if j == 0 else [], fontsize=8)
            ax.set_title(f"{dim} — {sset}", loc="left", fontsize=9.5)
            if i == nrow - 1:
                ax.set_xlabel(("change in " if mode == "delta" else "")
                              + METRICS[metric]
                              + (" at a 1000-pick budget" if metric == "ef" else ""))
    handles = [mpl.patches.Patch(color=c, label=l) for l, c in seen.items()]
    if handles:
        fig.legend(handles=handles, loc="lower center", ncol=min(2, len(handles)),
                   frameon=False, fontsize=8)
    fig.suptitle(f"{METRICS[metric]} by screen family"
                 + (f" — change vs {reference}" if mode == "delta" else ""),
                 x=0.01, ha="left", fontsize=11)
    fig.tight_layout(rect=(0, 0.035 + 0.030 * len(seen), 1, 0.965))
    plt.show()
    return pd.concat(kept, ignore_index=True) if kept else pd.DataFrame()

FAM_EF = fig_type_metric("ef")

In [ ]:
# the paired form, which is what to read when comparing arms rather than levels
fig_type_metric("ef", mode="delta");

# and one narrowed call, to show the vocabulary: two dimensions, two categories
fig_type_metric("ef", dims=["Perturbation", "Phenotype"],
                cats=["Knockout", "Activation", "Drug", "Fitness"]);

## Figure 10 — is the advantage real, or EF inflation from low-hit screens?

`EF = h / (b_eff * H/D) = precision / prevalence`, so **one hit on a screen with two hits in
19,000 genes is worth ~10 EF, while one hit on a screen with 1,500 hits is worth ~0.1**. A
handful of near-empty screens can therefore carry an entire headline, and the 218-screen fold
contains plenty of them — the curated 20-screen sets do not, because `num_hits_min: 50` was
applied when they were selected.

This figure answers the question directly, two ways.

**Left — where the change lives.** Paired change against the baseline, binned by hits
available. If a method only wins in the `1-9` and `10-49` bins, its headline is leverage.

**Right — does it survive?** The same mean change as the minimum-hits floor rises. A line that
decays toward zero was inflation; a line that holds is a real effect. The curated floor of 50
is marked.

The table underneath gives the number that settles it: **the share of the total change
contributed by screens below each floor.** If 5 screens out of 120 carry half the gain, that
belongs in any sentence quoting it.

In [ ]:
# --- Figure 10: hit-count leverage -------------------------------------------
# EF is precision/prevalence, so a hit is worth ~1/(b_eff * H/D) EF -- around 10 EF on a
# 2-hit screen against ~0.1 EF on a 1500-hit one. These two panels separate a real effect
# from one carried by a few near-empty screens.
HIT_BINS   = [(0, 10, "1-9"), (10, 50, "10-49"), (50, 200, "50-199"),
              (200, 1000, "200-999"), (1000, None, "1000+")]
HIT_FLOORS = [0, 10, 20, 50, 100, 200]
CURATED_FLOOR = 50          # the curated sets' own num_hits_min

def _paired(label, base, sset, metric, mask=None):
    """Paired per-screen difference, as a frame with `pub` kept for clustering."""
    a, b = arm_frame(label, sset), arm_frame(base, sset)
    j = a.merge(b, on=["sset", "screen"], suffixes=("", "_b"))
    if mask is not None:
        j = j[mask(j)]
    if j.empty:
        return None
    return j.assign(d=j[metric] - j[f"{metric}_b"])

def _mean_ci(j, col="d"):
    per = (j.groupby("pub")[col].mean().to_numpy() if _unit(j) == "pub"
           else j[col].to_numpy())
    if len(per) == 0:
        return np.nan, np.nan, np.nan, 0
    if len(per) < 2:
        return float(per.mean()), np.nan, np.nan, len(per)
    rng = np.random.default_rng(RNG_SEED)
    bs = per[rng.integers(0, len(per), (N_BOOT, len(per)))].mean(1)
    return (float(per.mean()), float(np.percentile(bs, 2.5)),
            float(np.percentile(bs, 97.5)), len(per))


def _share(part, tot, tot_abs):
    """Share of the total change, or NaN when that share is not interpretable.

    If positive and negative per-screen changes largely cancel, the total is near zero and
    a ratio to it explodes -- a bin can read 338% or -101%, which says nothing. The guard
    requires the net to be at least a fifth of the gross before a share is reported; the
    `sum_delta` column is always valid and is what to read when share is NaN."""
    if not np.isfinite(tot) or not np.isfinite(tot_abs) or tot_abs <= 1e-9:
        return np.nan
    if abs(tot) < 0.2 * tot_abs:
        return np.nan
    return 100 * part / tot


def hit_bin_table(metric="ef", select=None, base="AF alone", sset=None, bins=None):
    bins = bins or HIT_BINS
    avail = set(LONG.sset.unique())
    ssets = [sset] if sset else [s for s in FAMILY_SETS if s in avail]
    rows = []
    for st in ssets:
        for l in [x for x in pick(select) if x != base and not arm_frame(x, st).empty]:
            full = _paired(l, base, st, metric)
            tot = float(full.d.sum()) if full is not None else np.nan
            tot_abs = float(full.d.abs().sum()) if full is not None else np.nan
            for lo, hi, name in bins:
                j = _paired(l, base, st, metric,
                            mask=lambda f, lo=lo, hi=hi: (
                                (f.total_hits >= lo)
                                & ((f.total_hits < hi) if hi else True)))
                if j is None:
                    continue
                m, clo, chi, nclus = _mean_ci(j)
                rows.append(dict(sset=st, label=l, base=base, bin=name, lo_hits=lo,
                                 n=j.screen.nunique(), pubs=nclus,
                                 delta=m, lo=clo, hi=chi,
                                 sum_delta=float(j.d.sum()),
                                 share=_share(float(j.d.sum()), tot, tot_abs)))
    return pd.DataFrame(rows)


def floor_table(metric="ef", select=None, base="AF alone", sset=None, floors=None):
    floors = floors or HIT_FLOORS
    avail = set(LONG.sset.unique())
    ssets = [sset] if sset else [s for s in FAMILY_SETS if s in avail]
    rows = []
    for st in ssets:
        for l in [x for x in pick(select) if x != base and not arm_frame(x, st).empty]:
            full = _paired(l, base, st, metric)
            tot = float(full.d.sum()) if full is not None else np.nan
            tot_abs = float(full.d.abs().sum()) if full is not None else np.nan
            for f0 in floors:
                j = _paired(l, base, st, metric,
                            mask=lambda f, f0=f0: f.total_hits >= f0)
                if j is None:
                    continue
                m, clo, chi, nclus = _mean_ci(j)
                rows.append(dict(sset=st, label=l, base=base, floor=f0,
                                 n=j.screen.nunique(), pubs=nclus,
                                 delta=m, lo=clo, hi=chi,
                                 sum_delta=float(j.d.sum()),
                                 kept_share=_share(float(j.d.sum()), tot, tot_abs)))
    return pd.DataFrame(rows)


def fig_hit_floor(metric="ef", select=None, base="AF alone", ssets=None, bins=None,
                  floors=None, figsize=None):
    B = hit_bin_table(metric, select, base, bins=bins)
    F = floor_table(metric, select, base, floors=floors)
    if B.empty and F.empty:
        print("nothing to draw"); return
    ssets = ssets or sorted(set(B.sset) | set(F.sset))
    labels = [l for l in pick(select) if l != base and l in set(B.label) | set(F.label)]
    cmap = {l: PALETTE[i % len(PALETTE)] for i, l in enumerate(labels)}
    nrow = len(ssets)
    fig, axes = plt.subplots(nrow, 2, squeeze=False,
                             figsize=figsize or (11.0, 3.6 * nrow))
    names = [n for _, _, n in (bins or HIT_BINS)]
    for r, st in enumerate(ssets):
        # ---- left: change by hit-count bin
        ax = axes[r][0]
        S = B[B.sset == st]
        x = np.arange(len(names)); w = 0.8 / max(1, len(labels))
        for k, l in enumerate(labels):
            ti = S[S.label == l].set_index("bin")
            vals = [ti.delta.get(n, np.nan) for n in names]
            pos = x - 0.4 + w * (k + 0.5)
            ax.bar(pos, vals, width=w * 0.9, color=cmap[l], label=l,
                   edgecolor="black" if l == STANDARD else "none",
                   linewidth=1.4 if l == STANDARD else 0)
            lo = [ti.lo.get(n, np.nan) for n in names]
            hi = [ti.hi.get(n, np.nan) for n in names]
            if np.isfinite(lo).any():
                err = [[v - a if np.isfinite(a) else 0 for v, a in zip(vals, lo)],
                       [b - v if np.isfinite(b) else 0 for v, b in zip(vals, hi)]]
                ax.errorbar(pos, vals, yerr=err, fmt="none", ecolor="black",
                            elinewidth=0.8, capsize=2)
        ax.axhline(0, color="black", lw=1)
        ns = [int(S[S.bin == n].n.max()) if len(S[S.bin == n]) else 0 for n in names]
        ax.set_xticks(x); ax.set_xticklabels([f"{n}\nn={k}" for n, k in zip(names, ns)],
                                             fontsize=7.5)
        ax.set_xlabel("hits available in the screen")
        ax.set_ylabel(f"change in {METRICS.get(metric, metric)}")
        ax.set_title(f"{st} — where the change lives", loc="left", fontsize=9.5)
        # ---- right: the robustness curve
        ax = axes[r][1]
        G = F[F.sset == st]
        for k, l in enumerate(labels):
            g = G[G.label == l].sort_values("floor")
            if g.empty:
                continue
            ax.plot(g.floor, g.delta, "o-", color=cmap[l], lw=1.7, ms=5, label=l)
            if g.lo.notna().any():
                ax.fill_between(g.floor, g.lo, g.hi, color=cmap[l], alpha=0.14, lw=0)
            for _, rr in g.iterrows():
                ax.annotate(f"{rr.delta:+.2f}", (rr.floor, rr.delta), fontsize=6.5,
                            textcoords="offset points", xytext=(0, 7), ha="center")
        ax.axhline(0, color="black", lw=1)
        ax.axvline(CURATED_FLOOR, color=NEUTRAL, ls=(0, (4, 2)), lw=1.2)
        ax.annotate(f"curated floor ({CURATED_FLOOR})", (CURATED_FLOOR, ax.get_ylim()[1]),
                    fontsize=7, color="#666", rotation=90, va="top",
                    xytext=(3, -3), textcoords="offset points")
        ax.set_xlabel("minimum hits available (screens below this dropped)")
        ax.set_ylabel(f"mean change in {METRICS.get(metric, metric)}")
        ax.set_title(f"{st} — does it survive?", loc="left", fontsize=9.5)
    handles = [mpl.patches.Patch(color=cmap[l], label=l) for l in labels]
    fig.legend(handles=handles, loc="lower center", ncol=min(2, len(labels)),
               frameon=False, fontsize=8)
    fig.suptitle(f"EF leverage check — paired change vs {base}, "
                 f"{CLUSTER_BY}-clustered 95% CI", x=0.01, ha="left", fontsize=11)
    fig.tight_layout(rect=(0, 0.05 + 0.035 * len(labels), 1, 0.955))
    plt.show()
    return B, F

BINS_T, FLOOR_T = fig_hit_floor("ef")

In [ ]:
# The number that settles it: how much of the total change survives each floor.
# `sum_delta` is in EF-units and is ALWAYS valid. `kept_share` is a percentage of the
# overall change and is deliberately NaN when positive and negative per-screen changes
# largely cancel -- a ratio to a near-zero total reads 338% or -101% and means nothing.
_f = FLOOR_T.copy()
print("paired change as the minimum-hits floor rises "
      f"({CLUSTER_BY}-clustered 95% CI, * excludes zero):\n")
for st in sorted(_f.sset.unique()):
    for l in sorted(_f[_f.sset == st].label.unique()):
        g = _f[(_f.sset == st) & (_f.label == l)].sort_values("floor")
        print(f"  {st} — {l}  (vs {g.base.iloc[0]})")
        for _, r in g.iterrows():
            sig = "*" if np.isfinite(r.lo) and (r.lo > 0 or r.hi < 0) else " "
            sh = (f"  retains {r.kept_share:5.1f}% of the total change"
                  if np.isfinite(r.kept_share) else
                  "  (share not interpretable: the net change nearly cancels)")
            print(f"      >={int(r.floor):<4} n={int(r.n):>4} pubs={int(r.pubs):>3}  "
                  f"d={r.delta:+.3f} [{r.lo:+.3f},{r.hi:+.3f}]{sig}"
                  f"  sum={r.sum_delta:+8.1f}{sh}")
        print()

(_f[["sset", "label", "floor", "n", "pubs", "delta", "lo", "hi", "sum_delta",
     "kept_share"]].round(3))

## 5. Tables

Three CSVs beside the notebook. `compare_hsa_long.csv` is the whole `LONG` frame, so any cut
not drawn here can be made in a spreadsheet without re-running anything.

In [ ]:
# --- exports -----------------------------------------------------------------
OUT = RUNS / "analysis_tables"
OUT.mkdir(parents=True, exist_ok=True)

LONG.to_csv(OUT / "compare_hsa_long.csv", index=False)

summ = pd.concat([summary_table("all", metric=m).assign(metric=m)
                  for m in ("ef", "hits", "hit_rate", "recall")], ignore_index=True)
summ.to_csv(OUT / "compare_hsa_summary.csv", index=False)

# ci=False: with every arm x 4 dimensions x every screen set, bootstrapping each
# category would dominate the notebook's runtime. Per-category intervals come from
# fig_type_metric(..., mode="delta") on demand instead.
bytype = pd.concat([type_table("ef", "all", dim=d, ci=False).assign(metric="ef")
                    for d in DIMS], ignore_index=True)
bytype.to_csv(OUT / "compare_hsa_by_screen_type.csv", index=False)

head = pd.concat([headroom_table("all", dim=d) for d in DIMS], ignore_index=True)
# scoped to FAMILY_SETS, like the figure -- widen that list, not this line
head.to_csv(OUT / "compare_hsa_headroom.csv", index=False)

pd.concat([hit_bin_table("ef", "all").assign(kind="bin"),
           floor_table("ef", "all").assign(kind="floor")], ignore_index=True)\
  .to_csv(OUT / "compare_hsa_hit_leverage.csv", index=False)

print("wrote:")
for f in ("compare_hsa_long.csv", "compare_hsa_summary.csv",
          "compare_hsa_by_screen_type.csv", "compare_hsa_headroom.csv",
          "compare_hsa_hit_leverage.csv"):
    print("  ", OUT / f)
summ[summ.metric == "ef"].sort_values(["sset", "value"], ascending=[True, False]).head(30)

## 6. Adding the GLM-5 -> HSA arm later

This is the one you said you will run. Nothing in the figures needs changing — add the run,
then add **one registry row**.

**Step 1 — produce the run.** The GLM-5 warm starts exist only for `paper_validation` and
`paper_test`, and they are **replayed from disk**, so this costs nothing. `run_paper_test_hsa.py`
does not yet accept a warm start; the smallest change is to pass the recorded warm-start
history into `run_one_screen` the way `sweep-dmcd-hyb-glm5tu-*` does, with
`hsa_scorer=make_hsa_scorer(...)` and `af_weight=0`. Write it to
`output/runs/external/paper_test_glm5_hsa.json` in the same row format as the other external
files.

**Step 2 — register it.** If it lands as an external JSON:

```python
EXTERNAL_SOURCES.append(("paper_test", "paper_test_glm5_hsa.json", "real"))
ARM_WEIGHTS["GLM-5 +top-up -> DepMap + HSA"] = (0.0, 0.5, 1.0)
```

and give it a label that keeps the warm start visible, by passing `prefix=` through
`canon_label` — or if it lands as a sweep:

```python
SWEEP_SOURCES.append(("paper_test", "sweep-glm5tu-hsa-pt",
                      "GLM-5 +top-up -> DepMap + HSA", 0.0, 0.5, 1.0))
```

Then re-run the registry and master-table cells. It will appear in `list_arms()`, in every
figure that selects it, and in the weight sweep if you add more than one DepMap weight.

**Step 3 — the comparison that matters.** The question is whether HSA rescues the warm start
that AF + DepMap could not:

```python
fig_delta("ef", select=["glm5", "DepMap + HSA (dm=0.5) [real mean]"],
          base="AF + DepMap (dm=0.5)", ssets=["paper_test"])
```

**Expect it to lose.** The warm start costs −0.516 [−0.859, −0.199] on 3 of 20 screens against
AF + DepMap with no LLM (HANDOFF §4.61b), and HSA's own cold-start text prior is worth ~nothing
(§24.21). Two separate reasons to predict a negative, which is worth stating before the run
rather than after.

---

### Flags worth flipping

| to answer | set |
|---|---|
| fair curated-vs-fold comparison | `CURATED_CRITERIA = True` |
| what the paper reports | `CLUSTER_BY = "screen"` |
| include the degenerate small libraries | `GENOME_SCALE_ONLY = False` |
| does the real prior matter | `select=["real"]` vs `select=["hsa"]` |

Re-run the master-table cell after changing a filter — `LONG` is built once.

## 7. The pool axis, the LLM protocols, and every run in one table

Added 2026-11-04. Three things above this point are now known to be wrong or incomplete, and
this section exists to carry the corrections rather than silently patch them:

**(a) `LONG.ef` is NOT adjusted EF.** It is `n_hits_vs_random` = `hits/(1000·H/pool)`, taken from
`r["ef"]` for external rows and `fm["n_hits_vs_random"]` for sweeps. Cells 0/2/4 call it
"adjusted EF"; they are wrong. Canonical adjusted EF is `hits/(eff·H/D)` with `D` the screen's own
library and `eff = 1000 − n2` (off-library-but-in-pool picks forgiven). The two rank arms
*differently*: `DepMap + HSA dm.50` reads **5.863** against `AF + DepMap`'s **5.670** in the
notebook's units, and **5.472 vs 5.532** in canonical — a reversal. The last cell of this section
repairs `LONG` in place.

**(b) The candidate pool is a choice, and it decides that comparison.** The f2 pool (21,147 genes)
is not the native setting: AssayBench hands each screen its own library unless `universe_genes`
overrides it, and the paper's f2 table exists *"for fair LLM comparison"* so that model arms face
what an LLM faces. The refund that pool requires is **per-arm**: `AF + DepMap` sends ~85 picks per
1,000 off-library, `dm.50` ~38, so the refund discounts `AF + DepMap`'s denominator twice as hard.
With pool = own library nothing lands off-library, `n2 = n3 = 0`, and all three EF formulas
collapse to one number. Both pools are loaded below as a `pool` column.

**(c) The curated 20-screen panels were the minority view.** On own-library pools, `dm.50` beats
`AF + DepMap` on paper_test and paper_validation, loses on the 69-screen validation fold, and
loses by **0.66** on the 146-screen held-out test fold. Set size and the verdict move together.

Everything in this section is **self-contained**: it reads the JSONs directly and does not use
`LONG`, `ARM_WEIGHTS`, `pick()` or the figure helpers above, so the two halves cannot silently
alter each other. It needs only `ROOT`, `EXT`, `BUDGET`, `PALETTE`, `NEUTRAL`, `N_BOOT` and
`RNG_SEED` from the setup cell.

In [ ]:
# --- section 7 loader: pool axis + protocol axis ------------------------------
# Tidy frame RECENT, one row per (set, pool, protocol, arm, screen), carrying BOTH EF
# conventions so no figure below can be in the wrong one by accident.
import yaml

RECENT_ARMS = ["AF alone", "AF + DepMap", "HSA only (ours)",
               "DepMap + HSA dm.25", "DepMap + HSA dm.50"]
SHORT_ARM = {"AF alone": "AF", "AF + DepMap": "AF + DepMap", "HSA only (ours)": "HSA",
             "DepMap + HSA dm.25": "DepMap + HSA .25",
             "DepMap + HSA dm.50": "DepMap + HSA .50"}
ARM_ORDER = [SHORT_ARM[a] for a in RECENT_ARMS]
ARM_COLOR = dict(zip(ARM_ORDER, PALETTE))
BASE_ARM = "AF + DepMap"

# AGGREGATION UNIT -- say it out loud, because it changes every level and every shift.
#   "pub"    each publication contributes the mean of its screens (the notebook's
#            CLUSTER_BY default, and what every interval here is bootstrapped over)
#   "screen" every screen counts once -- what AF_DEPMAP_HSA_SUMMARY section 4.9's TABLES use
# They disagree, sometimes a lot: the validation fold's `AF + DepMap` pool shift is -0.256
# by screen and -0.410 by publication. Neither is wrong; quoting one as the other is.
RECENT_UNIT = "pub"

def agg_ef(df, metric="ef"):
    if df.empty:
        return np.nan
    if RECENT_UNIT == "pub":
        return float(df.groupby("pub")[metric].mean().mean())
    return float(df[metric].mean())

# (set, pool, protocol, LLM share of budget, [files])
# FILE ORDER MATTERS: the first file an arm appears in wins. The REAL-MEAN file is listed
# first everywhere, because every own-library run uses `banks_real_*` and comparing one
# against the mean=0 file mixes bank variants (it cost -0.10 EF on paper_test's HSA row).
RECENT_SOURCES = [
    ("paper_test",       "f2",      "cold",     0.000,
     ["paper_test_hsa_realmean.json", "paper_test_hsa.json"]),
    ("paper_test",       "library", "cold",     0.000, ["paper_test_hsa_libpool.json"]),
    ("paper_test",       "f2",      "retain25", 0.075, ["glm5_retain25_hsa.json"]),
    ("paper_test",       "f2",      "warm",     0.291, ["glm5_handoff_hsa.json"]),
    ("paper_validation", "f2",      "cold",     0.000,
     ["paper_validation_hsa_realmean.json", "paper_validation_hsa.json"]),
    ("paper_validation", "library", "cold",     0.000, ["paper_validation_hsa_libpool.json"]),
    ("validation fold",  "f2",      "cold",     0.000,
     ["validation_fold_hsa_REALCOMBINED.json"]),
    ("validation fold",  "library", "cold",     0.000, ["validation_fold_hsa_libpool.json"]),
    ("TEST fold",        "library", "cold",     0.000, ["test_fold_hsa_libpool.json"]),
    ("team11 external",  "library", "cold",     0.000, ["team11_hsa.json"]),
]

# publication = the clustering unit. team11's map is the PAPER dict of score_team11_hsa.py.
_SET_MANIFEST = {
    "paper_test":       ROOT / ".venv/lib/python3.11/site-packages/assaybench/data/"
                               "screen_sets/assayloop-test.yaml",
    "paper_validation": ROOT / ".venv/lib/python3.11/site-packages/assaybench/data/"
                               "screen_sets/assayloop-validation.yaml",
    "validation fold":  ROOT / "_reproduction/donor/validation_fold_218.yaml",
    "TEST fold":        ROOT / "_reproduction/donor/test_fold_334.yaml",
}
_TEAM11_PAPER = {
    "ABT_PARCB_dependency": "abt", "NGUYEN_AraC_resistant": "nguyen",
    "ORGANOID_WTSI-COLIVM_031_minLib": "organoid",
    "ORGANOID_WTSI-OESO_005_minLib": "organoid",
    "ORGANOID_WTSI-OVAR_018_minLib": "organoid",
    "SCHUKKEN_PAIRED_WGL_paper": "schukken", "SCHUKKEN_PAIRED_WGL_rank194": "schukken",
    "SHAN_HeLa_LDLRko": "shan", "SHAN_HeLa_lovastatin": "shan",
    "SHAN_HepG2_lovastatin": "shan", "TAU_CRISPRi_dec": "tau", "WILSON_MERCS_inc": "wilson",
}

_PM = {}
def pubmap(sset):
    if sset in _PM:
        return _PM[sset]
    if sset == "team11 external":
        _PM[sset] = dict(_TEAM11_PAPER)
        return _PM[sset]
    p = _SET_MANIFEST.get(sset)
    out = {}
    if p is not None and Path(p).exists():
        for s in yaml.safe_load(Path(p).read_text())["screens"]:
            out[s["dataset_name"]] = str(s["source_id"])
    _PM[sset] = out
    return out


def _rows_from(sset, pool, protocol, share, files):
    pm, seen, out, src = pubmap(sset), set(), [], {}
    for fn in files:
        p = EXT / fn
        if not p.exists():
            continue
        for r in json.loads(p.read_text()):
            arm = r["arm"]
            if arm not in RECENT_ARMS or (arm, r["screen"]) in seen:
                continue
            seen.add((arm, r["screen"]))
            src.setdefault(arm, fn)
            picks = int(r.get("picks", BUDGET))
            n_meas = int(r.get("n_measured", picks))
            n2 = picks - n_meas                      # off-library but inside the pool
            eff = BUDGET - n2                        # unfilled slots stay CHARGED
            prev = r["total_hits"] / r["n_genes"]    # the screen's own-library prevalence
            out.append(dict(
                sset=sset, pool=pool, protocol=protocol, llm_share=share,
                arm=SHORT_ARM[arm], raw_arm=arm, screen=r["screen"],
                pub=pm.get(r["screen"], "?" + r["screen"]), source=fn,
                hits=int(r["hits"]), picks=picks, n_measured=n_meas,
                n_genes=int(r["n_genes"]), total_hits=int(r["total_hits"]),
                ef=r["hits"] / (eff * prev),                       # CANONICAL adjusted EF
                ef_nvr=float(r.get("ef", np.nan)),                 # as stored (pool prevalence)
                hit_rate=r["hits"] / max(1, n_meas),
                rate_slots=r["hits"] / BUDGET,
                recall=r["hits"] / max(1, r["total_hits"])))
    return out, src


_rows, _prov, _missing = [], [], []
for sset, pool, protocol, share, files in RECENT_SOURCES:
    rs, src = _rows_from(sset, pool, protocol, share, files)
    if not rs:
        _missing.append((sset, pool, protocol, files))
        continue
    _rows += rs
    for arm, fn in src.items():
        _prov.append(dict(sset=sset, pool=pool, protocol=protocol, arm=arm, file=fn))
RECENT = pd.DataFrame(_rows)
PROV = pd.DataFrame(_prov)

if _missing:
    print("MISSING (skipped, everything else still draws):")
    for m in _missing:
        print("  ", m)

# The identity that makes the own-library runs unambiguous, checked rather than assumed.
_lib = RECENT[RECENT.pool == "library"]
_bad = _lib[(_lib.picks != BUDGET) | (_lib.n_measured != BUDGET)]
assert _bad.empty, f"own-library rows with off-library picks:\n{_bad.head()}"
_gap = (_lib.ef - _lib.hits / (BUDGET * _lib.total_hits / _lib.n_genes)).abs().max()
print(f"RECENT: {len(RECENT):,} rows, {RECENT.sset.nunique()} sets, "
      f"{RECENT.pool.nunique()} pools, {RECENT.protocol.nunique()} protocols")
print(f"own-library identity holds: max |canonical - hits/(1000*H/D)| = {_gap:.2e}")
print(RECENT.groupby(["sset", "pool", "protocol"])
      .agg(screens=("screen", "nunique"), pubs=("pub", "nunique"), arms=("arm", "nunique"))
      .to_string())

In [ ]:
# --- the master table: every run, every arm, both EF conventions ---------------
def cl_boot(vals_by_pub):
    """Publication-clustered bootstrap mean + 95% interval over {pub: [values]}."""
    k = list(vals_by_pub)
    if len(k) < 2:
        m = float(np.mean([v for vs in vals_by_pub.values() for v in vs])) if k else np.nan
        return m, np.nan, np.nan
    m = np.array([np.mean(vals_by_pub[x]) for x in k])
    rng = np.random.default_rng(RNG_SEED)
    bs = m[rng.integers(0, len(k), (N_BOOT, len(k)))].mean(1)
    return float(m.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def _paired_vs_base(g, metric="ef", base=BASE_ARM):
    """dEF against `base` within one (set, pool, protocol) block, clustered by publication."""
    b = g[g.arm == base].set_index("screen")
    if b.empty:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    out = {}
    wins = tot = 0
    dh = 0
    for r in g.itertuples():
        if r.screen not in b.index or r.arm == base:
            continue
        d = getattr(r, metric) - b.at[r.screen, metric]
        out.setdefault(r.pub, []).append(d)
        wins += d > 0
        tot += 1
        dh += r.hits - b.at[r.screen, "hits"]
    if not tot:
        return 0.0, np.nan, np.nan, 0, 0
    m, lo, hi = cl_boot(out)
    return m, lo, hi, dh, wins / tot


# How many screens a set has under BOTH pools. The validation fold is 218 under f2 but
# only 69 under own-library (`--min-genes 18000`: a 1,000-pick budget needs a library
# bigger than 1,000, and the fold's median is 2,318). So its two EF levels are NOT on the
# same screens and must not be read side by side -- `paired_n` says where that bites, and
# Figure 11 restricts to the overlap before drawing any arrow.
_pool_overlap = {}
for _s, _g in RECENT[RECENT.protocol == "cold"].groupby("sset"):
    _p = {p: set(gg.screen) for p, gg in _g.groupby("pool")}
    _pool_overlap[_s] = len(set.intersection(*_p.values())) if len(_p) > 1 else _g.screen.nunique()


def recent_master(metric="ef", base=BASE_ARM, export=True):
    rows = []
    for (sset, pool, proto), g in RECENT.groupby(["sset", "pool", "protocol"], sort=False):
        by_pub = {}
        for r in g.itertuples():
            by_pub.setdefault((r.arm, r.pub), []).append(getattr(r, metric))
        for arm in ARM_ORDER:
            ga = g[g.arm == arm]
            if ga.empty:
                continue
            vals = {p: v for (a, p), v in by_pub.items() if a == arm}
            m, lo, hi = cl_boot(vals)
            sub = g[g.arm.isin([arm, base])]
            d, dlo, dhi, dh, wr = _paired_vs_base(sub, metric, base) if arm != base \
                else (0.0, np.nan, np.nan, 0, np.nan)
            rows.append(dict(
                set=sset, pool=pool, protocol=proto, arm=arm,
                screens=ga.screen.nunique(), pubs=ga.pub.nunique(),
                paired_n=_pool_overlap.get(sset, np.nan),
                EF=m, EF_lo=lo, EF_hi=hi,
                EF_nvr=float(ga.groupby("pub").ef_nvr.mean().mean()),
                hits=int(ga.hits.sum()),
                avail=int(ga.groupby("screen").total_hits.first().sum()),
                rate_slots=float(ga.hits.sum()) / (ga.screen.nunique() * BUDGET),
                dEF=d, dEF_lo=dlo, dEF_hi=dhi, d_hits=dh, win_rate=wr))
    T = pd.DataFrame(rows)
    T["pct_avail"] = 100 * T.hits / T.avail.replace(0, np.nan)
    T["sig"] = np.where(T.dEF_lo.notna() & ((T.dEF_lo > 0) | (T.dEF_hi < 0)), "*", "")
    if export:
        out = RUNS / "analysis_tables" / "recent_master.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        T.to_csv(out, index=False)
        print("wrote", out)
    return T


MASTER = recent_master()
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 120)
print("\nEF = CANONICAL adjusted EF, publication-clustered. EF_nvr = the notebook's old unit.")
print("dEF is paired against", BASE_ARM, "within the same (set, pool, protocol) block.\n")
for _s, _n in sorted(_pool_overlap.items()):
    _tot = MASTER[MASTER.set == _s].screens.max()
    if _n != _tot:
        print(f"CAUTION  {_s}: {_tot} screens under f2 but only {_n} under own-library, so "
              f"those two EF levels are NOT the same screens. Figure 11 pairs them; this "
              f"table does not.")
print(f"\nEF above is {RECENT_UNIT}-level. Both conventions, own-library cold runs:")
_cmp = []
for (st_, arm), g in RECENT[(RECENT.pool == "library") & (RECENT.protocol == "cold")] \
        .groupby(["sset", "arm"]):
    _cmp.append(dict(set=st_, arm=arm, EF_pub=g.groupby("pub").ef.mean().mean(),
                     EF_screen=g.ef.mean()))
_CMP = pd.DataFrame(_cmp)
_CMP["gap"] = _CMP.EF_pub - _CMP.EF_screen
print(_CMP.round(3).to_string(index=False))
print("\nAF_DEPMAP_HSA_SUMMARY section 4.9's tables are the EF_screen column.")
print()
MASTER[["set", "pool", "protocol", "arm", "screens", "paired_n", "pubs", "EF", "EF_nvr",
        "hits", "pct_avail", "dEF", "dEF_lo", "dEF_hi", "d_hits", "win_rate", "sig"]].round(3)

In [ ]:
# Which file supplied each arm -- the guard against mixing mean=0 and real-mean banks.
PROV.pivot_table(index=["sset", "pool", "protocol"], columns="arm", values="file",
                 aggfunc="first")

### Figure 11 — the pool changes the answer, and by how much per arm

Each arm's canonical EF under the f2 pool against the same arm on its own library, same screens.
The arrow length is the refund being removed. Read the **family split**: AF-based arms send ~60–100
picks per 1,000 off-library and lose 0.19–0.28 EF; HSA arms send ~28–50 and lose ≤0.11 or gain.
Inside a family the order does not follow the count, so do not read it as monotone.

**Aggregation unit:** these cells use `RECENT_UNIT`, publication-level by default, matching `CLUSTER_BY` and every interval in this notebook. The tables in `AF_DEPMAP_HSA_SUMMARY.md` §4.9 are **screen-level**, so the shifts there read smaller (validation fold `AF + DepMap`: −0.256 by screen, −0.410 by publication). Set `RECENT_UNIT = "screen"` to reproduce the document's numbers exactly.

In [ ]:
# --- Figure 11: f2 -> own-library, per arm ------------------------------------
def fig_pool_shift(metric="ef", ssets=None, figsize=None):
    P = RECENT[RECENT.protocol == "cold"]
    pairs = []
    for sset in (ssets or ["paper_test", "paper_validation", "validation fold"]):
        g = P[P.sset == sset]
        if {"f2", "library"} - set(g.pool.unique()):
            continue
        shared = set(g[g.pool == "f2"].screen) & set(g[g.pool == "library"].screen)
        pairs.append((sset, g[g.screen.isin(shared)]))
    if not pairs:
        print("need both pools on at least one set"); return
    fig, axes = plt.subplots(1, len(pairs), squeeze=False,
                             figsize=figsize or (4.4 * len(pairs), 3.4))
    for ax, (sset, g) in zip(axes[0], pairs):
        for i, arm in enumerate(ARM_ORDER):
            a = g[(g.arm == arm) & (g.pool == "f2")]
            b = g[(g.arm == arm) & (g.pool == "library")]
            if a.empty or b.empty:
                continue
            x0, x1 = agg_ef(a, metric), agg_ef(b, metric)
            off = (a.picks - a.n_measured).mean()
            ax.annotate("", xy=(x1, i), xytext=(x0, i),
                        arrowprops=dict(arrowstyle="-|>", lw=1.6, color=ARM_COLOR[arm],
                                        shrinkA=0, shrinkB=0))
            ax.scatter([x0], [i], s=26, color="white", edgecolor=ARM_COLOR[arm], zorder=3)
            ax.scatter([x1], [i], s=52, color=ARM_COLOR[arm], zorder=3)
            ax.text(max(x0, x1) + 0.02, i, f"  {x1 - x0:+.3f}   ({off:.0f} off-lib)",
                    va="center", fontsize=7)
        ax.set_yticks(range(len(ARM_ORDER)))
        ax.set_yticklabels(ARM_ORDER if ax is axes[0][0] else [], fontsize=8)
        ax.set_xlabel("canonical adjusted EF")
        ax.set_title(f"{sset}  (n={g.screen.nunique()}, {g.pub.nunique()} pubs)", fontsize=9)
        ax.margins(x=0.22)
    fig.suptitle(f"hollow = f2 pool (refund ON)   solid = own library (nothing to "
             f"refund)   —  {RECENT_UNIT}-level means",
             y=1.02, fontsize=9)
    fig.tight_layout()
    plt.show()

fig_pool_shift()

### Figure 12 — four sets, one contrast, ordered by size

`dm.50` and the other arms against `AF + DepMap` on **own-library pools only**, so every point is
in the one unambiguous metric. Sets are ordered by screen count. The two curated 20-screen panels
sit above zero; the 69-screen validation fold and the 146-screen held-out test fold sit below it,
and the test fold is the only set where an interval clears zero against `AF + DepMap`.

In [ ]:
# --- Figure 12: own-library forest, four sets --------------------------------
def fig_forest(metric="ef", base=BASE_ARM, arms=None, figsize=(7.4, 5.6)):
    T = MASTER[(MASTER.pool == "library") & (MASTER.protocol == "cold")
               & (MASTER.arm != base)].copy()
    if arms:
        T = T[T.arm.isin(arms)]
    if T.empty:
        print("nothing to draw"); return
    order = (T.groupby("set").screens.max().sort_values().index.tolist())
    rows = []
    for s in order:
        for arm in ARM_ORDER:
            r = T[(T.set == s) & (T.arm == arm)]
            if not r.empty:
                rows.append(r.iloc[0])
    R = pd.DataFrame(rows).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(R))[::-1]
    ax.errorbar(R.dEF, y, xerr=[R.dEF - R.dEF_lo, R.dEF_hi - R.dEF], fmt="none",
                ecolor="black", elinewidth=0.9, capsize=2.5, zorder=2)
    ax.scatter(R.dEF, y, s=[70 if a == "DepMap + HSA .50" else 40 for a in R.arm],
               c=[ARM_COLOR[a] for a in R.arm],
               edgecolor=["black" if s == "*" else "white" for s in R.sig],
               linewidth=1.4, zorder=3)
    ax.axvline(0, color="black", lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{r.set}  ·  {r.arm}" for r in R.itertuples()], fontsize=8)
    for yi, r in zip(y, R.itertuples()):
        ax.text(ax.get_xlim()[1], yi, f"  {r.d_hits:+,} hits   {r.win_rate:.0%} won",
                va="center", fontsize=7, color="#444")
    ax.set_xlabel(f"change in canonical adjusted EF vs {base}  "
                  f"(publication-clustered 95% CI; black ring = excludes zero)")
    ax.set_title("own-library pools only — sets ordered by screen count (small at top)",
                 fontsize=9)
    ax.margins(x=0.30)
    fig.tight_layout()
    plt.show()
    return R

FOREST = fig_forest()

### Figure 13 — how much budget the LLM should be allowed to spend

paper_test, f2 pool, GLM-5 rounds 1–3 replayed from disk. Three protocols differ only in what
those rounds do with the model's proposals: **cold** 0% of the budget, **retain25** 7.5% (keep the
25 best-scoring proposals, the arm fills the other 75 — the teammate's rule), **warm** 29.1% (the
whole recorded round, shortfall lost). Rounds 4–10 are identical in all three. EF peaks at the
*cap*, not at the full warm start, for every arm — and the cap is also where hits peak.

In [ ]:
# --- Figure 13: LLM budget share, dose response ------------------------------
def fig_llm_dose(sset="paper_test", figsize=(9.0, 3.6)):
    G = RECENT[(RECENT.sset == sset) & (RECENT.pool == "f2")]
    protos = [p for p in ("cold", "retain25", "warm") if p in set(G.protocol)]
    if len(protos) < 2:
        print("need at least two protocols"); return
    shares = {p: G[G.protocol == p].llm_share.iloc[0] for p in protos}
    # only screens present in every protocol, so the lines are paired
    keep = set.intersection(*[set(G[G.protocol == p].screen) for p in protos])
    G = G[G.screen.isin(keep)]
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    for arm in ARM_ORDER:
        ga = G[G.arm == arm]
        if ga.empty:
            continue
        xs = [100 * shares[p] for p in protos]
        ef = [agg_ef(ga[ga.protocol == p]) for p in protos]
        hv = [ga[ga.protocol == p].hits.sum() for p in protos]
        for ax, ys in ((axes[0], ef), (axes[1], hv)):
            ax.plot(xs, ys, "o-", color=ARM_COLOR[arm], lw=1.7, ms=6, label=arm,
                    mec="white", mew=1.0)
    for ax, lab in ((axes[0], "canonical adjusted EF"), (axes[1], "total hits")):
        ax.set_xlabel("share of the 10x100 budget spent by the LLM (%)")
        ax.set_ylabel(lab)
        ax.set_xticks([100 * shares[p] for p in protos])
        ax.set_xticklabels([f"{p}\n{100 * shares[p]:.1f}%" for p in protos], fontsize=8)
    axes[1].yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=min(5, len(l)), frameon=False, fontsize=8)
    fig.suptitle(f"{sset}, {len(keep)} screens — GLM-5 replayed from disk, nothing billed",
                 fontsize=9, y=1.0)
    fig.tight_layout(rect=(0, 0.13, 1, 0.94))
    plt.show()

fig_llm_dose()

### Figure 14 — EF against hits, every run in one frame

The trade that defines this whole strand. Up is better EF, right is better hits. `AF + DepMap`
(ringed) sits up-and-left on the two folds; the HSA arms sit down-and-right almost everywhere.
Hollow markers are f2-pool runs, solid are own-library: an arm's two markers show the refund
moving it vertically while its hits barely change.

In [ ]:
# --- Figure 14: EF vs hits, pools overlaid -----------------------------------
def fig_tradeoff_pools(ssets=None, figsize=None):
    T = MASTER[MASTER.protocol == "cold"]
    ssets = ssets or [s for s in ["paper_test", "paper_validation", "validation fold",
                                  "TEST fold", "team11 external"] if s in set(T.set)]
    fig, axes = plt.subplots(1, len(ssets), squeeze=False,
                             figsize=figsize or (3.5 * len(ssets), 3.5))
    for ax, sset in zip(axes[0], ssets):
        g = T[T.set == sset]
        for r in g.itertuples():
            solid = r.pool == "library"
            ax.scatter(r.rate_slots * 100, r.EF, s=90 if r.arm == BASE_ARM else 60,
                       facecolor=ARM_COLOR[r.arm] if solid else "white",
                       edgecolor="black" if r.arm == BASE_ARM else ARM_COLOR[r.arm],
                       linewidth=1.6 if r.arm == BASE_ARM else 1.2, zorder=3, label=r.arm)
        for arm in ARM_ORDER:                        # join an arm's two pools
            two = g[g.arm == arm]
            if len(two) == 2:
                ax.plot(two.rate_slots * 100, two.EF, color=ARM_COLOR[arm], lw=0.8,
                        ls=(0, (3, 2)), zorder=2)
        ax.set_title(f"{sset}\n({int(g.screens.max())} screens)", fontsize=8.5)
        ax.set_xlabel("hits per 100 slots offered")
        if ax is axes[0][0]:
            ax.set_ylabel("canonical adjusted EF")
    h, l = axes[0][0].get_legend_handles_labels()
    seen, hh, ll = set(), [], []
    for a, b in zip(h, l):
        if b not in seen:
            seen.add(b); hh.append(a); ll.append(b)
    fig.legend(hh, ll, loc="lower center", ncol=min(5, len(ll)), frameon=False, fontsize=8)
    fig.suptitle("hollow = f2 pool, solid = own library; dashed line joins one arm's two pools",
                 fontsize=9, y=1.0)
    fig.tight_layout(rect=(0, 0.14, 1, 0.93))
    plt.show()

fig_tradeoff_pools()

### Figure 15 — the test-fold margin is concentrated, and that has to travel with it

`AF + DepMap` wins the 146-screen test fold by 0.66 EF over `dm.50`, but that average is not a
broad win: publication means split roughly in half, and a handful of publications supply the
margin. One (`36574441`, 20 screens) averages about −4.4 on its own. This is why `dm.50`'s
interval still reaches above zero despite a −0.66 mean, and why the honest phrasing is
"`AF + DepMap` wins on average, by a margin a minority of publications creates".

In [ ]:
# --- Figure 15: per-publication contribution, one set ------------------------
def fig_pub_concentration(sset="TEST fold", arm="DepMap + HSA .50", base=BASE_ARM,
                          metric="ef", figsize=(8.2, 4.6)):
    g = RECENT[(RECENT.sset == sset) & (RECENT.pool == "library")]
    a = g[g.arm == arm].set_index("screen")
    b = g[g.arm == base].set_index("screen")
    both = a.index.intersection(b.index)
    if not len(both):
        print(f"no shared screens for {arm!r} vs {base!r} on {sset}"); return
    d = pd.DataFrame({"pub": a.loc[both, "pub"],
                      "d": a.loc[both, metric] - b.loc[both, metric]})
    P = (d.groupby("pub").d.agg(["mean", "size"]).sort_values("mean")
         .rename(columns={"size": "screens"}))
    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(P))
    ax.bar(x, P["mean"], color=["#55A868" if v > 0 else "#C44E52" for v in P["mean"]],
           edgecolor="white", linewidth=0.4)
    for xi, (m, n) in enumerate(zip(P["mean"], P["screens"])):
        if n >= 5 or abs(m) > 1.5:
            ax.text(xi, m + (0.12 if m > 0 else -0.12), f"n={n}", ha="center",
                    va="bottom" if m > 0 else "top", fontsize=6.5)
    ax.axhline(0, color="black", lw=1)
    ax.axhline(P["mean"].mean(), color=NEUTRAL, ls=(0, (4, 2)), lw=1.3)
    ax.annotate(f"unweighted mean of publication means = {P['mean'].mean():+.3f}",
                (0, P["mean"].mean()), xytext=(4, 6), textcoords="offset points",
                fontsize=7.5, color="#555")
    up = int((P["mean"] > 0).sum())
    ax.set_xticks(x)
    ax.set_xticklabels(P.index, rotation=90, fontsize=6)
    ax.set_ylabel(f"{arm} minus {base}\ncanonical adjusted EF")
    ax.set_title(f"{sset}: {up} of {len(P)} publications favour {arm}, "
                 f"{len(both)} screens — green = favours {arm}", fontsize=9)
    fig.tight_layout()
    plt.show()
    return P

PUBC = fig_pub_concentration()

### The `LONG.ef` repair — opt in, then re-run Figures 1–10

`LONG.ef` is `n_hits_vs_random`, not adjusted EF (see (a) at the top of this section). This cell
rebuilds it as canonical adjusted EF and keeps the old column as `ef_nvr`, so the figures above
can be read in the project's metric. It is **idempotent** and **off by default**: set
`REPAIR_LONG_EF = True`, run it, then re-run whichever of Figures 1–10 you want in canonical units.

The shortcut `n2 = picks − n_measured` is exact only while every pick comes from the pool, which
holds for these files; for an **under-supplied LLM arm** the unfilled slots are charged and belong
in `n3`, not in the refund, so the cell refuses to touch rows with `picks < BUDGET` and names
them.

In [ ]:
# --- repair LONG.ef to canonical adjusted EF (opt-in, idempotent) -------------
REPAIR_LONG_EF = False          # set True, run, then re-run Figures 1-10

if not REPAIR_LONG_EF:
    print("REPAIR_LONG_EF is False -- LONG.ef is still n_hits_vs_random. "
          "Set it True and re-run this cell to switch Figures 1-10 to canonical EF.")
elif "ef_nvr" in LONG.columns:
    print("already repaired: LONG.ef is canonical, LONG.ef_nvr is the old unit")
else:
    short = LONG[LONG.picks < BUDGET]
    if len(short):
        print(f"!! {len(short)} under-supplied rows ({short.label.nunique()} arms) -- their "
              "unfilled slots are CHARGED, so `picks - n_measured` is not n2 for them. "
              "They keep the old value and are listed below; score those arms with "
              "scripts/diagnostics/score_glm5_handoff_hsa.py instead.")
        print(short.groupby(["sset", "label"]).size().to_string())
    LONG["ef_nvr"] = LONG.ef
    ok = LONG.picks >= BUDGET
    n2 = (LONG.picks - LONG.n_measured).clip(lower=0)
    canon = LONG.hits / ((BUDGET - n2) * LONG.total_hits / LONG.n_genes)
    LONG.loc[ok, "ef"] = canon[ok]
    METRICS["ef_nvr"] = "n_hits_vs_random (pool prevalence, full budget)"
    METRICS["ef"] = "canonical adjusted EF (own library, off-library picks forgiven)"
    moved = (LONG.loc[ok].assign(d=LONG.loc[ok, "ef"] - LONG.loc[ok, "ef_nvr"])
             .groupby(["sset", "label"]).d.mean().sort_values())
    print(f"repaired {int(ok.sum()):,} of {len(LONG):,} rows. Largest shifts:")
    print(pd.concat([moved.head(4), moved.tail(4)]).round(3).to_string())